# Main listening and metric evaluation

This notebook is part of the thesis notebook set.
It uses the shared prepared 70/30 speech/music split directory: `/content/drive/MyDrive/master/prepared_datasets/audio_70speech_30music_v1/splits`.

Purpose:
- This notebook renders listening examples and computes controlled metrics.
- Checkpoint prefixes and manual checkpoint paths are configuration fields, not fixed thesis assumptions.
- The model-specific training or evaluation logic is kept from the original notebook unless the configuration depended on an old data split.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

In [ ]:
# =========================
# Imports
# =========================
import os, sys, json, math, random, subprocess, gc
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from torch.nn.utils import spectral_norm
from IPython.display import Audio, display, Markdown

print("torch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass


In [ ]:
# =========================
# Main config
# =========================
DRIVE_ROOT = Path("/content/drive/MyDrive/master")
CHECKPOINTS_RUNS_DIR = DRIVE_ROOT / "checkpoints_runs"
PREPARED_DATASET_DIR = DRIVE_ROOT / "prepared_datasets" / "audio_70speech_30music_v1"
SPLIT_DIR = PREPARED_DATASET_DIR / "splits"
ADAPTER_RUNS_DIR = DRIVE_ROOT / "adapter_runs"
RUN_NAME = "main_listening_and_metric"
RUN_DIR = DRIVE_ROOT / "eval_runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Configurable listening clip.
# DOMAIN can be "speech" or "music". The split file is chosen from this.
LISTENING_DOMAIN = "speech"
LISTENING_INDEX = 4
LISTENING_START_SEC = 0.0
LISTENING_SEG_S_BY_DOMAIN = {"speech": 6.0, "music": 10.0}
LISTENING_MUSIC_DEFAULT_START_SEC = 20.0

# Render these cases for the chosen clip.
RUN_LISTENING_EVAL = True
LISTENING_INCLUDE_1X_INPAINT = True
LISTENING_STRETCH_SPEEDS = [0.75, 0.50]

# Backward-compatible exact-clip defaults used by older helper functions.
MANUAL_FILE_PATH = None
SPLIT_NAME = f"{LISTENING_DOMAIN}_test.txt"
SPLIT_INDEX = LISTENING_INDEX
K = 1
OFFSET = 0
SEG_S = float(LISTENING_SEG_S_BY_DOMAIN.get(LISTENING_DOMAIN, 6.0))

# BigVGAN / mel frontend defaults.
SR = 22050
BIGVGAN_MODEL_ID = "nvidia/bigvgan_v2_22khz_80band_fmax8k_256x"
BIGVGAN_USE_CUDA_KERNEL = False
N_FFT = 1024
HOP = 256
WIN = 1024
N_MELS = 80
FMIN = 0
FMAX = 8000
CENTER_MEL = False

# STFT defaults used for the STFT / hybrid branches.
CPLX_N_FFT = 1024
CPLX_HOP = 256
CPLX_WIN = 1024
CPLX_CENTER = True

# Blend-grid defaults for complex-mask models.
# Actual checkpoints override these from run_config["blend_stft"] when available.
BLEND_N_FFT = 1024
BLEND_HOP = 256
BLEND_WIN = 1024
BLEND_CENTER = True

# HF emphasis.
HF_START_FRAC = 0.45
HF_END_GAIN = 8.0
HF_RAMP_POWER = 2.0
MASK_DILATE = 2
TDIFF_MASK_DILATE = 3
HF_D_START_FRAC = 0.45
HF_D_MASK_DILATE = 2

# Hybrid defaults.
G_GROUPS = 8
G_BASE = 24
CPLX_BASE = 24
FUSION_BASE = 24
D_BASE = 32
PRIOR_RADIUS = 3
PRIOR_BOUNDARY_GAIN = 1.5
PRIOR_HF_POWER = 1.25
GATE_TEMP = 2.0

# Complex-mask model defaults.
# Checkpoint run_config overrides these.
FUSION_IN_CH = 7
MASK_TEMP = 1.0
MASK_REAL_MAX = 1.2
MASK_IMAG_SCALE = 0.75
MASK_MODE = "complex"

# Output controls.
SHOW_MEL_PLOTS = True
SHOW_WAVE_ZOOM = False
WAVE_ZOOM_MS = 80
SAVE_AUDIO_FILES = True
COMPUTE_ROUNDTRIP_METRICS = False

# Numeric evaluation controls.
RUN_NUMERIC_TEST_EVAL = True
RUN_EXPLORATORY_NUMERIC_TEST_EVAL = False  # set True for the secondary thesis-variant metric table
MAX_SPEECH_METRIC_FILES = None             # None means every speech test file
MAX_MUSIC_METRIC_FILES = None              # None means every music test file
SPEECH_METRIC_SEG_S = 4.0
MUSIC_METRIC_SEG_S = 10.0
METRIC_START_S = 0.0
SAVE_EVERY_N_CASES = 25

# Training-log plotting.
PLOT_TRAINING_LOGS = False
TRAIN_METRICS_TO_PLOT = [
    "loss_total", "loss_recon", "loss_hf", "loss_tdiff",
    "loss_delta", "loss_gate", "loss_gate_align", "loss_roundtrip_hf",
    "lossG", "lossD", "loss_mel", "loss_adv", "loss_fm",
    "loss_rt_hf", "loss_rt_tdiff",
    "base_recon", "ref_recon", "base_hf", "ref_hf", "base_tdiff", "ref_tdiff",
    "imp_hf_pct", "imp_pct", "avg_gate", "borrow_hf",
    "loss_logstft_hf", "loss_complex", "loss_phase", "loss_wav_mrstft",
    "loss_mel_recon", "loss_mel_hf", "loss_mel_tdiff",
    "loss_mask_real", "loss_mask_imag", "loss_blend_delta",
    "mel_logstft_hf", "stft_logstft_hf", "blend_logstft_hf",
    "mel_mel_hf", "stft_mel_hf", "blend_mel_hf",
    "alpha_mean", "beta_abs_mean", "blend_delta_mag",
    "loss_mel_lowmid", "base_logmag", "ref_logmag", "base_complex", "ref_complex",
]
TRAIN_SMOOTH_WINDOW = 1
TRAIN_PLOT_NCOLS = 3

# Main final listening models.
MAIN_LISTENING_MODEL_NAMES = [
    "Full-skip 2D U-Net",
    "Complex-STFT CRN",
    "Residual-mined adapter alpha 1.00",
    "Residual-mined adapter alpha 1.50",
    "Residual-mined adapter alpha 2.00",
]

# Optional secondary metric models. These are development/exploratory variants,
# not part of the compact listening set.
EXPLORATORY_METRIC_MODEL_NAMES = [
    "BigVGAN-aux moderate-GAN plain 2D U-Net",
    "High-frequency dual-discriminator plain 2D U-Net",
    "Discriminator-family plain 2D U-Net",
    "Supervisor HFRAW plain 2D U-Net",
    "Supervisor HFRT plain 2D U-Net",
    "Complex STFT U-Net v20",
    "Early mel-complex fusion v23",
    "Fresh end-to-end hybrid v25",
    "Mel-dominant STFT assist v35",
    "Mel-dominant STFT assist v35 stronger",
    "Mel-space gated fusion v50",
    "Complex-mask blend v41",
    "Complex-mask blend verified/fixed v42-44",
    "Complex-mask broad mel-anchor v45",
    "Real-mask broad mel-anchor ablation v46",
]

# Checkpoint specs. The final main table uses the compact set above.
MODEL_SPECS = [
    dict(name="Full-skip 2D U-Net", family="fullskip_plain2d",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),

    dict(name="Complex-STFT CRN", family="complexstft_crn",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),

    dict(name="BigVGAN-aux moderate-GAN plain 2D U-Net", family="plain2d",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),
    dict(name="High-frequency dual-discriminator plain 2D U-Net", family="plain2d",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),
    dict(name="Discriminator-family plain 2D U-Net", family="plain2d",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),
    dict(name="Supervisor HFRAW plain 2D U-Net", family="plain2d",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),
    dict(name="Supervisor HFRT plain 2D U-Net", family="plain2d",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),

    dict(name="Complex STFT U-Net v20", family="complexstft",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),

    dict(name="Early mel-complex fusion v23", family="meldom_stft_assist",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),
    dict(name="Fresh end-to-end hybrid v25", family="hybrid",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),
    dict(name="Mel-dominant STFT assist v35", family="meldom_stft_assist",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),
    dict(name="Mel-dominant STFT assist v35 stronger", family="meldom_stft_assist",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),
    dict(name="Mel-space gated fusion v50", family="meldom_stft_assist",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),

    dict(name="Complex-mask blend v41", family="complexmask",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),
    dict(name="Complex-mask blend verified/fixed v42-44", family="complexmask",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),
    dict(name="Complex-mask broad mel-anchor v45", family="complexmask",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),
    dict(name="Real-mask broad mel-anchor ablation v46", family="complexmask",
         prefix=None, manual_ckpt=None, manual_log=None, enabled=True),
]

V1_RUN_NAME = "adapter_run_name_here"
V1_RUN_DIR = ADAPTER_RUNS_DIR / V1_RUN_NAME
MANUAL_V1_ADAPTER_CKPT = None  # Example: Path("/content/drive/MyDrive/master/checkpoints_runs/<run_name>/latest.pt")
RESIDUAL_ALPHA_SWEEP = [1.00, 1.50, 2.00]
OPTIONAL_RESIDUAL_ALPHA_VALUES = [1.25, 1.75]

for alpha in RESIDUAL_ALPHA_SWEEP:
    MODEL_SPECS.append(dict(
        name=f"Residual-mined adapter alpha {alpha:.2f}",
        family="residual_adapter",
        prefix=None,
        manual_ckpt=MANUAL_V1_ADAPTER_CKPT,
        manual_log=None,
        enabled=True,
        alpha=float(alpha),
    ))

RUN_PITCH_DIAGNOSTICS = False
PITCH_MODEL_NAME = None

required = [DRIVE_ROOT, CHECKPOINTS_RUNS_DIR, SPLIT_DIR, ADAPTER_RUNS_DIR, RUN_DIR]
for p in required:
    print(p, "exists:", p.exists())
print("RUN_DIR:", RUN_DIR)
print("Listening split/index:", SPLIT_NAME, SPLIT_INDEX)
print("Main listening models:", MAIN_LISTENING_MODEL_NAMES)
print("Secondary exploratory metrics enabled:", RUN_EXPLORATORY_NUMERIC_TEST_EVAL)


In [ ]:

# =========================
# BigVGAN repo / imports
# =========================
BIGVGAN_REPO = Path("/content/BigVGAN")
if not BIGVGAN_REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/NVIDIA/BigVGAN.git", "/content/BigVGAN"], check=True)

if str(BIGVGAN_REPO) not in sys.path:
    sys.path.insert(0, str(BIGVGAN_REPO))

import bigvgan
from meldataset import mel_spectrogram as bigvgan_mel_spectrogram

bigvgan_G = None

def ensure_bigvgan_loaded():
    global bigvgan_G
    if bigvgan_G is not None:
        return bigvgan_G

    print("Loading BigVGAN:", BIGVGAN_MODEL_ID)
    bigvgan_G = bigvgan.BigVGAN._from_pretrained(
        model_id=BIGVGAN_MODEL_ID,
        revision="main",
        cache_dir=None,
        force_download=False,
        proxies=None,
        resume_download=False,
        local_files_only=False,
        token=None,
        map_location=str(device),
        strict=False,
        use_cuda_kernel=BIGVGAN_USE_CUDA_KERNEL,
    ).to(device).eval()

    for p in bigvgan_G.parameters():
        p.requires_grad_(False)

    return bigvgan_G

def wav_to_bigvgan_mel(wav_bt: torch.Tensor):
    if wav_bt.ndim == 1:
        wav_bt = wav_bt.unsqueeze(0)
    return bigvgan_mel_spectrogram(
        wav_bt,
        n_fft=N_FFT,
        num_mels=N_MELS,
        sampling_rate=SR,
        hop_size=HOP,
        win_size=WIN,
        fmin=FMIN,
        fmax=FMAX,
        center=CENTER_MEL,
    )

def mel_to_wave_bigvgan(mel_bt: torch.Tensor):
    Gv = ensure_bigvgan_loaded()
    y = Gv(mel_bt)
    return y.squeeze(1) if (y.ndim == 3 and y.shape[1] == 1) else y

print("BigVGAN repo ready at:", BIGVGAN_REPO)



# =========================
# Dataset / loaders
# =========================
def read_list(p: Path):
    assert p.exists(), f"Missing list: {p}"
    lines = [ln.strip().strip('"').strip("'") for ln in p.read_text().splitlines()]
    return [Path(ln) for ln in lines if ln]

def safe_load_wav_mono(path: Path, target_sr: int) -> torch.Tensor:
    path = Path(path)
    try:
        wav, sr = torchaudio.load(str(path))
        wav = wav.mean(dim=0)
    except Exception:
        data, sr = sf.read(str(path), dtype="float32", always_2d=True)
        wav = torch.from_numpy(data.T).mean(dim=0)

    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)

    wav = wav.to(torch.float32)
    wav = wav / (wav.abs().max() + 1e-8)
    wav = (0.95 * wav).clamp(-1.0, 1.0)
    return wav



In [ ]:

# =========================
# Helpers: masks, weights, mel/STFT corruption, losses
# =========================
def build_freq_weights(n_bins: int, start_frac: float, end_gain: float, power: float, device=None):
    idx = torch.linspace(0.0, 1.0, steps=n_bins, device=device)
    ramp = ((idx - start_frac).clamp_min(0.0) / max(1e-6, 1.0 - start_frac)) ** power
    w = 1.0 + (end_gain - 1.0) * ramp
    return w.view(1, n_bins, 1)

HF_FREQ_WEIGHTS = build_freq_weights(N_MELS, HF_START_FRAC, HF_END_GAIN, HF_RAMP_POWER, device=device)
CPLX_FREQ_WEIGHTS = build_freq_weights(CPLX_N_FFT // 2 + 1, HF_START_FRAC, HF_END_GAIN, HF_RAMP_POWER, device=device)

def dilate_mask_time(mask: torch.Tensor, radius: int):
    if radius <= 0:
        return mask
    return F.max_pool2d(mask, kernel_size=(1, 2 * radius + 1), stride=1, padding=(0, radius))

def expand_mask(mask: torch.Tensor, n_bins: int, radius: int = 0):
    m = dilate_mask_time(mask, radius)
    return m[:, 0].expand(-1, n_bins, -1)

def masked_mean(x: torch.Tensor, mask_ft: torch.Tensor, freq_weights=None, eps: float = 1e-8):
    z = x.abs()
    if freq_weights is not None:
        z = z * freq_weights
        denom = (mask_ft * freq_weights).sum()
    else:
        denom = mask_ft.sum()
    return (z * mask_ft).sum() / (denom + eps)

def masked_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    m = expand_mask(mask, pred.shape[1], radius=mask_dilate)
    return masked_mean(pred - tgt, m, freq_weights=freq_weights)

def masked_tdiff_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    dp = pred[..., 1:] - pred[..., :-1]
    dt = tgt[..., 1:] - tgt[..., :-1]
    m = expand_mask(mask, pred.shape[1], radius=mask_dilate)[..., 1:]
    return masked_mean(dp - dt, m, freq_weights=freq_weights)

def masked_complex_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    m = expand_mask(mask, pred.shape[1], radius=mask_dilate)
    diff = torch.complex(pred.real - tgt.real, pred.imag - tgt.imag)
    return masked_mean(diff, m, freq_weights=freq_weights)

def masked_logmag_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    pm = torch.log1p(pred.abs())
    tm = torch.log1p(tgt.abs())
    return masked_l1(pm, tm, mask, freq_weights=freq_weights, mask_dilate=mask_dilate)

def linear_time_inpaint_mel(mel: torch.Tensor, k: int, offset: int):
    B, F, T = mel.shape
    step = k + 1
    mel_interp = mel.clone()
    mask = torch.zeros((B, 1, 1, T), device=mel.device, dtype=torch.float32)
    for left in range(offset, T - step, step):
        right = left + step
        for j in range(1, step):
            t = left + j
            alpha = j / step
            mel_interp[:, :, t] = (1.0 - alpha) * mel[:, :, left] + alpha * mel[:, :, right]
            mask[:, :, :, t] = 1.0
    return mel_interp, mask

def stft_complex(wav_bt: torch.Tensor):
    window = torch.hann_window(CPLX_WIN, device=wav_bt.device)
    return torch.stft(
        wav_bt,
        n_fft=CPLX_N_FFT,
        hop_length=CPLX_HOP,
        win_length=CPLX_WIN,
        window=window,
        center=CPLX_CENTER,
        return_complex=True,
    )

def istft_complex(X_bft: torch.Tensor, length: int):
    window = torch.hann_window(CPLX_WIN, device=X_bft.device)
    return torch.istft(
        X_bft,
        n_fft=CPLX_N_FFT,
        hop_length=CPLX_HOP,
        win_length=CPLX_WIN,
        window=window,
        center=CPLX_CENTER,
        length=length,
    )

def linear_time_inpaint_complex(X: torch.Tensor, k: int, offset: int):
    B, Freq, T = X.shape
    step = k + 1
    Xr = X.real.clone()
    Xi = X.imag.clone()
    mask = torch.zeros((B, 1, 1, T), device=X.device, dtype=torch.float32)
    for left in range(offset, T - step, step):
        right = left + step
        for j in range(1, step):
            t = left + j
            alpha = j / step
            Xr[:, :, t] = (1.0 - alpha) * X.real[:, :, left] + alpha * X.real[:, :, right]
            Xi[:, :, t] = (1.0 - alpha) * X.imag[:, :, left] + alpha * X.imag[:, :, right]
            mask[:, :, :, t] = 1.0
    return torch.complex(Xr, Xi), mask

def make_shared_pair(wav_bt: torch.Tensor, k_choices=(1, 2), randomize_offset=True):
    k = int(random.choice(list(k_choices)))
    step = k + 1
    offset = int(random.randrange(step)) if randomize_offset else 0

    mel_real = wav_to_bigvgan_mel(wav_bt)
    mel_interp, mask_mel = linear_time_inpaint_mel(mel_real, k=k, offset=offset)

    X_real = stft_complex(wav_bt)
    X_interp, mask_cplx = linear_time_inpaint_complex(X_real, k=k, offset=offset)

    return {
        "wav": wav_bt,
        "mel_real": mel_real,
        "mel_interp": mel_interp,
        "mask_mel": mask_mel,
        "X_real": X_real,
        "X_interp": X_interp,
        "mask_cplx": mask_cplx,
        "k": k,
        "offset": offset,
    }

def make_hf_prior(mask_mel: torch.Tensor, n_mels: int, radius: int = PRIOR_RADIUS, boundary_gain=None, hf_power=None):
    base = expand_mask(mask_mel, n_mels, radius=0)
    wide = expand_mask(mask_mel, n_mels, radius=radius)
    boundary = (wide - base).clamp_min(0.0)

    if boundary_gain is None:
        boundary_gain = PRIOR_BOUNDARY_GAIN
    if hf_power is None:
        hf_power = PRIOR_HF_POWER

    hf = HF_FREQ_WEIGHTS.expand(wide.shape[0], -1, wide.shape[-1])
    hf_norm = ((hf - 1.0) / max(1e-6, (HF_END_GAIN - 1.0))).clamp(0.0, 1.0)
    hf_shape = hf_norm ** hf_power

    region = (base + boundary_gain * boundary).clamp(0.0, 1.0 + boundary_gain)
    return (region * hf_shape).clamp(0.0, 1.0 + PRIOR_BOUNDARY_GAIN)

def make_gate_target(mel_main_hat: torch.Tensor, mel_complex_hat: torch.Tensor, mel_real: torch.Tensor, prior: torch.Tensor):
    err_main = (mel_main_hat - mel_real).abs()
    err_cplx = (mel_complex_hat - mel_real).abs()
    gain = (err_main - err_cplx).clamp_min(0.0) * prior
    den = gain.amax(dim=(1, 2), keepdim=True).clamp_min(1e-6)
    target = (gain / den).clamp(0.0, 1.0)
    return target ** 0.75

def make_adv_focus_view(mel_bt: torch.Tensor, mask_mel: torch.Tensor):
    start_bin = int(round(HF_D_START_FRAC * mel_bt.shape[1]))
    mel_hf = mel_bt[:, start_bin:, :]
    mask_hf = expand_mask(mask_mel, mel_bt.shape[1], radius=HF_D_MASK_DILATE)[:, start_bin:, :]
    return mel_hf * mask_hf, mask_hf

def get_roundtrip_views(mel_hat: torch.Tensor, mel_real: torch.Tensor, mask_mel: torch.Tensor, with_grad_fake: bool = True):
    sb = min(ROUNDTRIP_SUBBATCH, mel_hat.shape[0])
    mel_hat_sb = mel_hat[:sb]
    mel_real_sb = mel_real[:sb]
    mask_sb = mask_mel[:sb]

    if ADV_VIEW_MODE == "raw_mel":
        mel_view_hat = mel_hat_sb
        mel_view_real = mel_real_sb
    else:
        if with_grad_fake:
            wav_hat = mel_to_wave_bigvgan(mel_hat_sb)
        else:
            with torch.no_grad():
                wav_hat = mel_to_wave_bigvgan(mel_hat_sb)

        with torch.no_grad():
            wav_real = mel_to_wave_bigvgan(mel_real_sb)

        mel_view_hat = wav_to_bigvgan_mel(wav_hat)
        mel_view_real = wav_to_bigvgan_mel(wav_real)

    T = min(mel_view_hat.shape[-1], mel_view_real.shape[-1], mask_sb.shape[-1])
    mel_view_hat = mel_view_hat[..., :T]
    mel_view_real = mel_view_real[..., :T]
    mask_sb = mask_sb[..., :T]

    return mel_view_hat, mel_view_real, mask_sb


def _match_frames(mel_bt: torch.Tensor, target_T: int):
    cur_T = int(mel_bt.shape[-1])
    if cur_T == target_T:
        return mel_bt
    if cur_T > target_T:
        return mel_bt[..., :target_T]
    return F.pad(mel_bt, (0, target_T - cur_T))

def roundtrip_mel_from_mel(mel_bt: torch.Tensor):
    wav_bt = mel_to_wave_bigvgan(mel_bt)
    mel_rt = wav_to_bigvgan_mel(wav_bt)
    return _match_frames(mel_rt, mel_bt.shape[-1])

def stft_complex_cfg(wav_bt: torch.Tensor, n_fft: int, hop: int, win: int, center: bool):
    window = torch.hann_window(win, device=wav_bt.device)
    return torch.stft(
        wav_bt, n_fft=n_fft, hop_length=hop, win_length=win,
        window=window, center=center, return_complex=True,
    )

def istft_complex_cfg(X_bft: torch.Tensor, n_fft: int, hop: int, win: int, center: bool, length: int):
    """Robust ISTFT wrapper.

    PyTorch can throw a window-overlap-add error for Hann windows with center=False,
    especially when we create new stretched STFT lengths. The training notebooks used
    different center settings across experiments, so for evaluation we first try the
    requested setting and then fall back to center=True if this specific COLA/WOLA
    error appears.
    """
    window = torch.hann_window(win, device=X_bft.device)

    def _call(center_flag: bool):
        return torch.istft(
            X_bft, n_fft=n_fft, hop_length=hop, win_length=win,
            window=window, center=center_flag, length=length,
        )

    try:
        return _call(center)
    except RuntimeError as e:
        msg = str(e)
        if ("window overlap add" in msg or "overlap add" in msg) and center is False:
            print("[ISTFT fallback] center=False caused a window-overlap-add error; retrying center=True for evaluation.")
            return _call(True)
        raise

def linear_time_inpaint_complex_cfg(X: torch.Tensor, k: int, offset: int):
    B, Freq, T = X.shape
    step = k + 1
    Xr = X.real.clone()
    Xi = X.imag.clone()
    mask = torch.zeros((B, 1, 1, T), device=X.device, dtype=torch.float32)
    if k == 0:
        return torch.complex(Xr, Xi), mask
    for left in range(offset, T - step, step):
        right = left + step
        for j in range(1, step):
            t = left + j
            alpha = j / step
            Xr[:, :, t] = (1.0 - alpha) * X.real[:, :, left] + alpha * X.real[:, :, right]
            Xi[:, :, t] = (1.0 - alpha) * X.imag[:, :, left] + alpha * X.imag[:, :, right]
            mask[:, :, :, t] = 1.0
    return torch.complex(Xr, Xi), mask

def trim_to_common_length(*xs):
    lens = [x.shape[-1] for x in xs if x is not None]
    T = min(lens)
    out = []
    for x in xs:
        out.append(None if x is None else x[..., :T])
    return out

def plain_mask_to_ft(mask: torch.Tensor, n_bins: int):
    return mask.expand(-1, n_bins, -1)

def exact_summary_row(model_name: str, family: str, out_kind: str, mel_hat: torch.Tensor, mel_real: torch.Tensor, mask_mel: torch.Tensor, extra=None):
    mel_base, mel_hat, mel_real, mask_mel = trim_to_common_length(CLIP_CTX["mel_interp"], mel_hat, mel_real, mask_mel)
    base_recon = masked_l1(mel_base, mel_real, mask_mel).item()
    ref_recon  = masked_l1(mel_hat, mel_real, mask_mel).item()
    base_hf = masked_l1(mel_base, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).item()
    ref_hf  = masked_l1(mel_hat, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).item()
    base_tdiff = masked_tdiff_l1(mel_base, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE).item()
    ref_tdiff  = masked_tdiff_l1(mel_hat, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE).item()

    if globals().get("COMPUTE_ROUNDTRIP_METRICS", False):
        mel_rt_real = roundtrip_mel_from_mel(mel_real)
        mel_rt_hat  = roundtrip_mel_from_mel(mel_hat)
        mel_rt_hat, mel_rt_real, mask_rt = trim_to_common_length(mel_rt_hat, mel_rt_real, mask_mel)
        rt_hf = masked_l1(mel_rt_hat, mel_rt_real, mask_rt, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).item()
        rt_tdiff = masked_tdiff_l1(mel_rt_hat, mel_rt_real, mask_rt, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE).item()
    else:
        rt_hf = np.nan
        rt_tdiff = np.nan

    row = dict(
        model=model_name,
        family=family,
        output_kind=out_kind,
        base_recon=base_recon, ref_recon=ref_recon,
        base_hf=base_hf, ref_hf=ref_hf,
        base_tdiff=base_tdiff, ref_tdiff=ref_tdiff,
        roundtrip_hf=rt_hf, roundtrip_tdiff=rt_tdiff,
        recon_improvement_pct=100.0 * (base_recon - ref_recon) / max(base_recon, 1e-8),
        hf_improvement_pct=100.0 * (base_hf - ref_hf) / max(base_hf, 1e-8),
        tdiff_improvement_pct=100.0 * (base_tdiff - ref_tdiff) / max(base_tdiff, 1e-8),
    )
    if extra:
        row.update(extra)
    return row


# =========================
# Extra helpers for complex-mask# =========================
# Extra helpers for complex-mask mel/STFT fusion checkpoints
# =========================
def match_frames_tensor(x: torch.Tensor, T: int):
    if x.shape[-1] > T:
        return x[..., :T]
    if x.shape[-1] < T:
        return F.pad(x, (0, T - x.shape[-1]))
    return x

def match_mask_frames(mask: torch.Tensor, T: int):
    if mask.shape[-1] > T:
        return mask[..., :T]
    if mask.shape[-1] < T:
        return F.pad(mask, (0, T - mask.shape[-1]))
    return mask

def trim_complex_to_common(*xs):
    T = min(x.shape[-1] for x in xs)
    return [x[..., :T] for x in xs]

def match_wav_length(wav_bt: torch.Tensor, target_len: int):
    if wav_bt.ndim == 1:
        wav_bt = wav_bt.unsqueeze(0)
    if wav_bt.shape[-1] > target_len:
        wav_bt = wav_bt[..., :target_len]
    elif wav_bt.shape[-1] < target_len:
        wav_bt = F.pad(wav_bt, (0, target_len - wav_bt.shape[-1]))
    return wav_bt

def safe_logmag_complex(X: torch.Tensor, eps: float = 1e-5):
    return torch.log(X.abs().clamp_min(eps))

def make_stft_prior_eval(mask_stft: torch.Tensor, n_bins: int, freq_weights: torch.Tensor,
                         radius: int, boundary_gain: float, hf_power: float,
                         stft_hf_end_gain: float):
    # mask_stft shape: (B, 1, 1, T)
    base = expand_mask(mask_stft, n_bins, radius=0)
    wide = expand_mask(mask_stft, n_bins, radius=radius)
    boundary = (wide - base).clamp_min(0.0)

    fw = freq_weights.to(mask_stft.device).expand(base.shape[0], -1, base.shape[-1])
    hf_norm = ((fw - 1.0) / max(1e-6, stft_hf_end_gain - 1.0)).clamp(0.0, 1.0)
    hf_shape = hf_norm ** hf_power

    region = (base + boundary_gain * boundary).clamp(0.0, 1.0)
    return (region * hf_shape).clamp(0.0, 1.0)

def build_complexmask_fusion_features(X_mel: torch.Tensor, X_stft: torch.Tensor,
                                      prior: torch.Tensor, mask_stft: torch.Tensor,
                                      fusion_in_ch: int = 7):
    X_mel, X_stft = trim_complex_to_common(X_mel, X_stft)
    T = X_mel.shape[-1]
    prior = match_frames_tensor(prior, T)
    mask_ft = expand_mask(match_mask_frames(mask_stft, T), X_mel.shape[1], radius=0)

    lm_mel = safe_logmag_complex(X_mel)
    lm_stft = safe_logmag_complex(X_stft)
    lm_diff = lm_stft - lm_mel

    phase_diff = torch.angle(X_stft) - torch.angle(X_mel)
    cos_d = torch.cos(phase_diff)
    sin_d = torch.sin(phase_diff)

    feat = torch.stack([lm_mel, lm_stft, lm_diff, cos_d, sin_d, prior, mask_ft], dim=1)
    feat[:, 0:3] = feat[:, 0:3].clamp(-12.0, 4.0)
    assert feat.ndim == 4 and feat.shape[1] == fusion_in_ch, f"bad fusion feature shape: {tuple(feat.shape)}"
    return feat, prior, mask_ft

def blend_with_complex_mask_eval(X_mel: torch.Tensor, X_stft: torch.Tensor, C: torch.Tensor, prior: torch.Tensor):
    X_mel, X_stft = trim_complex_to_common(X_mel, X_stft)
    T = X_mel.shape[-1]
    C = match_frames_tensor(C, T)
    prior = match_frames_tensor(prior, T)
    return X_mel + prior * C * (X_stft - X_mel)

def extract_state_dict(ckpt, preferred_keys=("G", "model", "model_state_dict", "state_dict", "generator")):
    if isinstance(ckpt, dict):
        for k in preferred_keys:
            if k in ckpt and isinstance(ckpt[k], dict):
                return ckpt[k]
        if all(torch.is_tensor(v) for v in ckpt.values()):
            return ckpt
    raise KeyError(f"Could not find model state dict. Available keys: {list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt)}")

def cfg_get(cfg, keys, default=None):
    for key in keys:
        if isinstance(key, tuple):
            cur = cfg
            ok = True
            for part in key:
                if isinstance(cur, dict) and part in cur:
                    cur = cur[part]
                else:
                    ok = False
                    break
            if ok:
                return cur
        else:
            if isinstance(cfg, dict) and key in cfg:
                return cfg[key]
    return default

def infer_base_from_state_dict(sd, default=24):
    for key in ["enc1.net.0.net.0.weight", "enc1.block.0.weight"]:
        if key in sd:
            return int(sd[key].shape[0])
    for k, v in sd.items():
        if k.endswith("weight") and getattr(v, "ndim", 0) == 4:
            return int(v.shape[0])
    return default


In [ ]:
# =========================
# Models: mel branch, complex branch, fusion gate, HF discriminator
# =========================
def _valid_groups(ch: int, requested: int):
    for g in [requested, 8, 4, 2, 1]:
        if ch % g == 0:
            return g
    return 1

class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, groups=8, act="lrelu"):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        act_layer = nn.LeakyReLU(0.2, inplace=True) if act == "lrelu" else nn.SiLU(inplace=True)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p),
            nn.GroupNorm(g, out_ch),
            act_layer,
        )
    def forward(self, x):
        return self.net(x)

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8, act="lrelu"):
        super().__init__()
        self.net = nn.Sequential(
            ConvGNAct(in_ch, out_ch, groups=groups, act=act),
            ConvGNAct(out_ch, out_ch, groups=groups, act=act),
        )
    def forward(self, x):
        return self.net(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=(2,2), groups=8, act="lrelu"):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        act_layer = nn.LeakyReLU(0.2, inplace=True) if act == "lrelu" else nn.SiLU(inplace=True)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1),
            nn.GroupNorm(g, out_ch),
            act_layer,
            ConvGNAct(out_ch, out_ch, groups=groups, act=act),
        )
    def forward(self, x):
        return self.net(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, groups=8, act="lrelu"):
        super().__init__()
        self.fuse = DoubleConv(in_ch + skip_ch, out_ch, groups=groups, act=act)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.fuse(x)

class MelBranchUNet2D(nn.Module):
    def __init__(self, n_mels, base=24, use_mask=True, groups=8):
        super().__init__()
        self.use_mask = use_mask
        c_in = 1 + (1 if use_mask else 0)
        self.enc1 = DoubleConv(c_in, base, groups=groups)
        self.enc2 = DownBlock(base, base*2, stride=(2,2), groups=groups)
        self.enc3 = DownBlock(base*2, base*4, stride=(2,2), groups=groups)
        self.enc4 = DownBlock(base*4, base*4, stride=(1,2), groups=groups)
        self.bot  = DoubleConv(base*4, base*4, groups=groups)
        self.up3 = UpBlock(base*4, base*4, base*4, groups=groups)
        self.up2 = UpBlock(base*4, base*4, base*2, groups=groups)
        self.up1 = UpBlock(base*2, base*2, base, groups=groups)
        self.out_conv = nn.Conv2d(base, 1, kernel_size=3, padding=1)
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, mel_interp: torch.Tensor, mask: torch.Tensor):
        x = mel_interp.unsqueeze(1)
        if self.use_mask:
            mask2d = mask.unsqueeze(2).expand(-1, 1, mel_interp.shape[1], -1)
            x = torch.cat([x, mask2d], dim=1)
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        b  = self.bot(s4)
        u3 = self.up3(b, s4)
        u2 = self.up2(u3, s3)
        u1 = self.up1(u2, s2)
        u0 = F.interpolate(u1, size=s1.shape[-2:], mode="bilinear", align_corners=False) + s1
        delta = self.out_conv(u0).squeeze(1)
        return delta

class ComplexConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class ComplexBranchUNet2D(nn.Module):
    def __init__(self, in_ch=3, out_ch=2, base=24, groups=8):
        super().__init__()
        self.enc1 = ComplexConvGNAct(in_ch, base, groups)
        self.enc2 = ComplexConvGNAct(base, base*2, groups)
        self.enc3 = ComplexConvGNAct(base*2, base*4, groups)
        self.bot  = ComplexConvGNAct(base*4, base*8, groups)
        self.dec3 = ComplexConvGNAct(base*8 + base*4, base*4, groups)
        self.dec2 = ComplexConvGNAct(base*4 + base*2, base*2, groups)
        self.dec1 = ComplexConvGNAct(base*2 + base, base, groups)
        self.out = nn.Conv2d(base, out_ch, kernel_size=3, padding=1)
        self.pool = nn.AvgPool2d(kernel_size=2)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))
        xb = self.bot(self.pool(x3))
        y3 = F.interpolate(xb, size=x3.shape[-2:], mode="bilinear", align_corners=False)
        y3 = self.dec3(torch.cat([y3, x3], dim=1))
        y2 = F.interpolate(y3, size=x2.shape[-2:], mode="bilinear", align_corners=False)
        y2 = self.dec2(torch.cat([y2, x2], dim=1))
        y1 = F.interpolate(y2, size=x1.shape[-2:], mode="bilinear", align_corners=False)
        y1 = self.dec1(torch.cat([y1, x1], dim=1))
        return self.out(y1)

def pack_complex_input(X_interp: torch.Tensor, mask: torch.Tensor):
    xr = X_interp.real.unsqueeze(1)
    xi = X_interp.imag.unsqueeze(1)
    xm = mask.expand(-1, 1, X_interp.shape[1], -1)
    return torch.cat([xr, xi, xm], dim=1)

class FusionUNet2D(nn.Module):
    def __init__(self, in_ch=5, base=24, groups=8):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base, groups=groups)
        self.enc2 = DownBlock(base, base*2, stride=(2,2), groups=groups)
        self.enc3 = DownBlock(base*2, base*4, stride=(2,2), groups=groups)
        self.enc4 = DownBlock(base*4, base*4, stride=(1,2), groups=groups)
        self.bot  = DoubleConv(base*4, base*4, groups=groups)
        self.up3 = UpBlock(base*4, base*4, base*4, groups=groups)
        self.up2 = UpBlock(base*4, base*4, base*2, groups=groups)
        self.up1 = UpBlock(base*2, base*2, base, groups=groups)
        self.gate_head = nn.Conv2d(base, 1, kernel_size=3, padding=1)
        self.delta_head = nn.Conv2d(base, 1, kernel_size=3, padding=1)
        nn.init.zeros_(self.gate_head.weight); nn.init.zeros_(self.gate_head.bias)
        nn.init.zeros_(self.delta_head.weight); nn.init.zeros_(self.delta_head.bias)

    def forward(self, mel_interp, mel_main, mel_complex, prior, mask):
        x = torch.stack([
            mel_interp,
            mel_main,
            mel_complex,
            prior,
            mask.expand(-1, mel_interp.shape[1], -1),
        ], dim=1)
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        b  = self.bot(s4)
        u3 = self.up3(b, s4)
        u2 = self.up2(u3, s3)
        u1 = self.up1(u2, s2)
        u0 = F.interpolate(u1, size=s1.shape[-2:], mode="bilinear", align_corners=False) + s1
        gate_logits = self.gate_head(u0).squeeze(1)
        gate = float(globals().get("GATE_MAX", 1.0)) * torch.sigmoid(GATE_TEMP * gate_logits)
        delta = self.delta_head(u0).squeeze(1)
        return gate, delta


# Backward-compatible aliases used by the different training notebooks/checkpoints.
PlainMelRefinerUNet2D = MelBranchUNet2D
ComplexSTFTUNet2D = ComplexBranchUNet2D

class FullSkipPlainMelRefinerUNet2D(nn.Module):
    """Retained full-skip mel 2D U-Net used by the final full-skip checkpoints."""
    def __init__(self, n_mels: int, base: int = 24, use_mask: bool = True, groups: int = 8):
        super().__init__()
        self.n_mels = n_mels
        self.use_mask = use_mask
        c_in = 1 + (1 if use_mask else 0)
        self.enc1 = DoubleConv(c_in,       base,     groups=groups)
        self.enc2 = DownBlock(base,        base*2,   stride=(2, 2), groups=groups)
        self.enc3 = DownBlock(base*2,      base*4,   stride=(2, 2), groups=groups)
        self.enc4 = DownBlock(base*4,      base*4,   stride=(1, 2), groups=groups)
        self.bot  = DoubleConv(base*4,     base*4,   groups=groups)
        self.up3 = UpBlock(base*4, base*4, base*4, groups=groups)
        self.up2 = UpBlock(base*4, base*4, base*2, groups=groups)
        self.up1 = UpBlock(base*2, base*2, base,   groups=groups)
        self.final_fuse = DoubleConv(base + base, base, groups=groups)
        self.out_conv = nn.Conv2d(base, 1, kernel_size=3, padding=1)
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, mel_interp: torch.Tensor, mask: torch.Tensor):
        x = mel_interp.unsqueeze(1)
        if self.use_mask:
            mask2d = mask.unsqueeze(2).expand(-1, 1, mel_interp.shape[1], -1)
            x = torch.cat([x, mask2d], dim=1)
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        b = self.bot(s4)
        u3 = self.up3(b,  s4)
        u2 = self.up2(u3, s3)
        u1 = self.up1(u2, s2)
        u0 = F.interpolate(u1, size=s1.shape[-2:], mode="bilinear", align_corners=False)
        fused = self.final_fuse(torch.cat([u0, s1], dim=1))
        return self.out_conv(fused).squeeze(1)


def mel_model_class_candidates(label: str = ""):
    """Try the expected architecture first, then the other one as fallback."""
    text = str(label).lower()
    if "full-skip" in text or "fullskip" in text:
        return [FullSkipPlainMelRefinerUNet2D, MelBranchUNet2D]
    return [MelBranchUNet2D, FullSkipPlainMelRefinerUNet2D]

class HFDisc(nn.Module):
    def __init__(self, in_ch=2, base=32):
        super().__init__()
        self.blocks = nn.ModuleList([
            nn.Sequential(spectral_norm(nn.Conv2d(in_ch, base, 5, stride=2, padding=2)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(spectral_norm(nn.Conv2d(base, base*2, 5, stride=2, padding=2)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(spectral_norm(nn.Conv2d(base*2, base*4, 5, stride=2, padding=2)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(spectral_norm(nn.Conv2d(base*4, base*4, 3, stride=1, padding=1)), nn.LeakyReLU(0.2, inplace=True)),
        ])
        self.out = spectral_norm(nn.Conv2d(base*4, 1, 3, stride=1, padding=1))

    def forward(self, x):
        feats = []
        h = x
        for blk in self.blocks:
            h = blk(h)
            feats.append(h)
        logit = self.out(h)
        return logit, feats




class TemporalBiGRUBottleneck(nn.Module):
    """BiGRU over the time axis at the U-Net bottleneck."""
    def __init__(self, channels: int, freq_bins: int, hidden: int = 256, layers: int = 1, dropout: float = 0.1):
        super().__init__()
        self.channels = channels
        self.freq_bins = freq_bins
        self.input_size = channels * freq_bins
        self.gru = nn.GRU(
            input_size=self.input_size,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if layers > 1 else 0.0,
        )
        self.proj = nn.Linear(2 * hidden, self.input_size)
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(self.input_size)

    def forward(self, x):
        B, C, Fq, T = x.shape
        if Fq != self.freq_bins:
            raise RuntimeError(f"Temporal bottleneck expected freq_bins={self.freq_bins}, got {Fq}. Check N_FFT/pooling.")
        seq = x.permute(0, 3, 1, 2).contiguous().view(B, T, C * Fq)
        y, _ = self.gru(seq)
        y = self.proj(y)
        y = self.norm(seq + self.drop(y))
        y = y.view(B, T, C, Fq).permute(0, 2, 3, 1).contiguous()
        return y

class ComplexSTFTCRNUNet2D(nn.Module):
    """DCCRN-lite inspired complex STFT model."""
    def __init__(self, in_ch=3, out_ch=2, base=16, groups=8, bottleneck_freq_bins=64,
                 crn_hidden=256, crn_layers=1, crn_dropout=0.1):
        super().__init__()
        self.enc1 = ComplexConvGNAct(in_ch, base, groups)
        self.enc2 = ComplexConvGNAct(base, base*2, groups)
        self.enc3 = ComplexConvGNAct(base*2, base*4, groups)
        self.bot  = ComplexConvGNAct(base*4, base*8, groups)
        self.temporal = TemporalBiGRUBottleneck(
            channels=base*8,
            freq_bins=bottleneck_freq_bins,
            hidden=crn_hidden,
            layers=crn_layers,
            dropout=crn_dropout,
        )
        self.dec3 = ComplexConvGNAct(base*8 + base*4, base*4, groups)
        self.dec2 = ComplexConvGNAct(base*4 + base*2, base*2, groups)
        self.dec1 = ComplexConvGNAct(base*2 + base, base, groups)
        self.out = nn.Conv2d(base, out_ch, kernel_size=3, padding=1)
        self.pool = nn.AvgPool2d(kernel_size=2)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))
        xb = self.bot(self.pool(x3))
        xb = self.temporal(xb)
        y3 = F.interpolate(xb, size=x3.shape[-2:], mode="bilinear", align_corners=False)
        y3 = self.dec3(torch.cat([y3, x3], dim=1))
        y2 = F.interpolate(y3, size=x2.shape[-2:], mode="bilinear", align_corners=False)
        y2 = self.dec2(torch.cat([y2, x2], dim=1))
        y1 = F.interpolate(y2, size=x1.shape[-2:], mode="bilinear", align_corners=False)
        y1 = self.dec1(torch.cat([y1, x1], dim=1))
        return self.out(y1)

class ComplexMaskFusionUNet2D(nn.Module):
    """
    Complex-mask fusion model used by notebooks 44/45/46.

    Inputs are seven STFT-grid feature maps:
      logmag(mel route), logmag(STFT route), logmag difference,
      cos phase diff, sin phase diff, prior, mask.

    Output is C = alpha + i beta.
    """
    def __init__(self, in_ch=7, base=24, groups=8,
                 mask_temp=1.0, mask_real_max=1.2, mask_imag_scale=0.75,
                 mask_mode="complex"):
        super().__init__()
        self.mask_temp = float(mask_temp)
        self.mask_real_max = float(mask_real_max)
        self.mask_imag_scale = float(mask_imag_scale)
        self.mask_mode = str(mask_mode)

        self.enc1 = DoubleConv(in_ch,       base,     groups=groups)
        self.enc2 = DownBlock(base,         base*2,   stride=(2,2), groups=groups)
        self.enc3 = DownBlock(base*2,       base*4,   stride=(2,2), groups=groups)
        self.enc4 = DownBlock(base*4,       base*4,   stride=(1,2), groups=groups)
        self.bot  = DoubleConv(base*4,      base*4,   groups=groups)

        self.up3 = UpBlock(base*4, base*4, base*4, groups=groups)
        self.up2 = UpBlock(base*4, base*4, base*2, groups=groups)
        self.up1 = UpBlock(base*2, base*2, base,   groups=groups)

        self.real_head = nn.Conv2d(base, 1, kernel_size=3, padding=1)
        self.imag_head = nn.Conv2d(base, 1, kernel_size=3, padding=1)

    def forward(self, feat_bctf):
        s1 = self.enc1(feat_bctf)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        b  = self.bot(s4)
        u3 = self.up3(b, s4)
        u2 = self.up2(u3, s3)
        u1 = self.up1(u2, s2)
        u0 = F.interpolate(u1, size=s1.shape[-2:], mode="bilinear", align_corners=False)
        u0 = u0 + s1

        real_raw = self.real_head(u0).squeeze(1)
        imag_raw = self.imag_head(u0).squeeze(1)

        alpha = self.mask_real_max * torch.sigmoid(self.mask_temp * real_raw)
        if self.mask_mode == "real_only":
            beta = torch.zeros_like(alpha)
        else:
            beta = self.mask_imag_scale * torch.tanh(imag_raw)

        C = torch.complex(alpha, beta)
        return C, alpha, beta


In [ ]:

# =========================
# Checkpoint discovery + exact clip
# =========================
def find_latest_ckpt(prefix: str):
    cands = sorted(CHECKPOINTS_RUNS_DIR.glob(f"{prefix}_*/latest.pt"))
    cands = [p for p in cands if p.exists()]
    return cands[-1] if cands else None

def resolve_log_path(spec, ckpt_path):
    if spec.get("manual_log", None):
        p = Path(spec["manual_log"])
        return p if p.exists() else None
    if ckpt_path is None:
        return None
    p = Path(ckpt_path).parent / "train_log.csv"
    return p if p.exists() else None

resolved_specs = []
for spec in MODEL_SPECS:
    if not spec.get("enabled", True):
        continue
    ckpt = spec.get("manual_ckpt", None)
    if ckpt is None:
        ckpt = find_latest_ckpt(spec["prefix"])
    ckpt = Path(ckpt) if ckpt is not None else None
    found = ckpt is not None and ckpt.exists()
    log_path = resolve_log_path(spec, ckpt if found else None)
    resolved_specs.append({**spec, "ckpt_path": ckpt, "found": found, "log_path": log_path})

display(pd.DataFrame([{
    "name": s["name"],
    "family": s["family"],
    "prefix": s["prefix"],
    "found": s["found"],
    "ckpt_path": None if s["ckpt_path"] is None else str(s["ckpt_path"]),
    "log_path": None if s["log_path"] is None else str(s["log_path"]),
} for s in resolved_specs]))
print("Resolved", sum(1 for s in resolved_specs if s["found"]), "checkpoints out of", len(resolved_specs), "enabled specs.")
print("If this is 0, check that Google Drive is mounted and that CHECKPOINTS_RUNS_DIR points to the right folder.")


def resolve_eval_path():
    if MANUAL_FILE_PATH:
        p = Path(MANUAL_FILE_PATH)
        assert p.exists(), f"MANUAL_FILE_PATH not found: {p}"
        return p
    split_path = SPLIT_DIR / SPLIT_NAME
    items = read_list(split_path)
    assert len(items) > SPLIT_INDEX, f"SPLIT_INDEX {SPLIT_INDEX} out of range for {split_path}"
    return items[SPLIT_INDEX]

EVAL_PATH = resolve_eval_path()
print("EVAL_PATH:", EVAL_PATH)

# Build exact clip context once.
@torch.no_grad()
def build_clip_context(path: Path):
    w = safe_load_wav_mono(path, SR)[: int(SR * SEG_S)].to(device)
    wb = w.unsqueeze(0)
    mel_real = wav_to_bigvgan_mel(wb)
    mel_interp, mask_mel = linear_time_inpaint_mel(mel_real, k=K, offset=OFFSET)
    wav_real_vocoder = mel_to_wave_bigvgan(mel_real)[0]
    wav_interp = mel_to_wave_bigvgan(mel_interp)[0]
    return dict(
        path=path,
        wav_real=w,
        wav_real_vocoder=wav_real_vocoder,
        wav_batch=wb,
        mel_real=mel_real,
        mel_interp=mel_interp,
        mask_mel=mask_mel,
        wav_interp=wav_interp,
    )

CLIP_CTX = build_clip_context(EVAL_PATH)
print("Clip context ready.")


In [ ]:
# =========================
# Full-skip U-Net and residual-mined adapter runners
# =========================
def torch_load_any(path, map_location="cpu"):
    try:
        return torch.load(str(path), map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(str(path), map_location=map_location)


def _mask_bt(mask_mel: torch.Tensor):
    if mask_mel.ndim == 4:
        return mask_mel[:, :, 0, :]
    return mask_mel


def _expand_mask_bt(mask_bt: torch.Tensor, n_mels: int):
    if mask_bt.ndim == 2:
        mask_bt = mask_bt.unsqueeze(1)
    return mask_bt.expand(-1, n_mels, -1)


FULLSKIP_CACHE = {}
ADAPTER_CACHE = {}


def _extract_state_from_ckpt(ck):
    for key in ["G", "model", "generator", "state_dict", "model_state_dict", "G_state_dict"]:
        if isinstance(ck, dict) and key in ck and isinstance(ck[key], dict):
            return ck[key]
    if isinstance(ck, dict) and any(torch.is_tensor(v) for v in ck.values()):
        return ck
    raise KeyError("Could not find a state dict in checkpoint")


def _cfg_value(ck, names, default=None):
    cfgs = []
    if isinstance(ck, dict):
        for key in ["run_config", "config", "model_config", "bigvgan_frontend_config"]:
            if isinstance(ck.get(key), dict):
                cfgs.append(ck[key])
        cfgs.append(ck)
    for cfg in cfgs:
        for name in names:
            if isinstance(name, tuple):
                cur = cfg
                ok = True
                for part in name:
                    if isinstance(cur, dict) and part in cur:
                        cur = cur[part]
                    else:
                        ok = False
                        break
                if ok:
                    return cur
            elif name in cfg:
                return cfg[name]
    return default


def load_fullskip_for_spec(spec):
    key = str(spec["ckpt_path"])
    if key in FULLSKIP_CACHE:
        return FULLSKIP_CACHE[key]
    ck = torch_load_any(spec["ckpt_path"], map_location="cpu")
    state = _extract_state_from_ckpt(ck)
    frontend = ck.get("bigvgan_frontend_config", {}) if isinstance(ck, dict) else {}
    rc = ck.get("run_config", {}) if isinstance(ck, dict) else {}
    n_mels = int(frontend.get("num_mels", rc.get("n_mels", N_MELS)))
    base = int(_cfg_value(ck, ["G_base", "g_base", "base", ("arch", "g_base")], G_BASE))
    groups = int(_cfg_value(ck, ["G_groups", "g_groups", "groups", ("arch", "g_groups")], G_GROUPS))
    use_mask = bool(_cfg_value(ck, ["G_use_mask", "g_use_mask", "use_mask", ("arch", "g_use_mask")], True))
    model = FullSkipPlainMelRefinerUNet2D(n_mels=n_mels, base=base, use_mask=use_mask, groups=groups).to(device)
    model.load_state_dict(state, strict=True)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    FULLSKIP_CACHE[key] = (model, ck)
    print(f"Loaded full-skip model: {spec['ckpt_path']}")
    return FULLSKIP_CACHE[key]


@torch.no_grad()
def run_fullskip_model(spec):
    model, ck = load_fullskip_for_spec(spec)
    mel_interp = CLIP_CTX["mel_interp"]
    mask_mel = CLIP_CTX["mask_mel"]
    mask_bt = _mask_bt(mask_mel)
    delta = model(mel_interp, mask_bt)
    mel_hat = mel_interp + _expand_mask_bt(mask_bt, mel_interp.shape[1]) * delta
    wav_hat = mel_to_wave_bigvgan(mel_hat)[0]
    row = exact_summary_row(
        model_name=spec["name"],
        family=spec["family"],
        out_kind="BigVGAN",
        mel_hat=mel_hat,
        mel_real=CLIP_CTX["mel_real"],
        mask_mel=mask_mel,
        extra=dict(step=int(ck.get("step", -1)) if isinstance(ck, dict) else -1),
    )
    return dict(
        name=spec["name"],
        family=spec["family"],
        out_label=f'{spec["name"]} -> BigVGAN',
        audio=wav_hat.detach().cpu().numpy(),
        mel_hat=mel_hat[0].detach().cpu(),
        summary=row,
    )


class AdapterConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, groups=8, dropout=0.0):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
        ]
        if dropout and dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


class AdapterUp(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, groups=8, dropout=0.0):
        super().__init__()
        self.conv = nn.Sequential(
            AdapterConvGNAct(in_ch + skip_ch, out_ch, groups=groups, dropout=dropout),
            AdapterConvGNAct(out_ch, out_ch, groups=groups, dropout=dropout),
        )
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class MissingFrameAdapterUNet(nn.Module):
    """Residual adapter architecture."""
    def __init__(self, in_ch=4, base=48, groups=8, dropout=0.03):
        super().__init__()
        self.e0 = nn.Sequential(
            AdapterConvGNAct(in_ch, base, groups=groups, dropout=dropout),
            AdapterConvGNAct(base, base, groups=groups, dropout=dropout),
        )
        self.e1 = nn.Sequential(
            AdapterConvGNAct(base, base*2, s=(2,2), groups=groups, dropout=dropout),
            AdapterConvGNAct(base*2, base*2, groups=groups, dropout=dropout),
        )
        self.e2 = nn.Sequential(
            AdapterConvGNAct(base*2, base*4, s=(2,2), groups=groups, dropout=dropout),
            AdapterConvGNAct(base*4, base*4, groups=groups, dropout=dropout),
        )
        self.e3 = nn.Sequential(
            AdapterConvGNAct(base*4, base*6, s=(2,2), groups=groups, dropout=dropout),
            AdapterConvGNAct(base*6, base*6, groups=groups, dropout=dropout),
        )
        self.bot = nn.Sequential(
            AdapterConvGNAct(base*6, base*6, k=3, p=1, groups=groups, dropout=dropout),
            nn.Conv2d(base*6, base*6, kernel_size=3, padding=2, dilation=2),
            nn.GroupNorm(_valid_groups(base*6, groups), base*6),
            nn.SiLU(inplace=True),
            nn.Conv2d(base*6, base*6, kernel_size=3, padding=4, dilation=4),
            nn.GroupNorm(_valid_groups(base*6, groups), base*6),
            nn.SiLU(inplace=True),
        )
        self.u2 = AdapterUp(base*6, base*4, base*4, groups=groups, dropout=dropout)
        self.u1 = AdapterUp(base*4, base*2, base*2, groups=groups, dropout=dropout)
        self.u0 = AdapterUp(base*2, base, base, groups=groups, dropout=dropout)
        self.out = nn.Conv2d(base, 1, kernel_size=3, padding=1)

    def forward(self, x):
        s0 = self.e0(x)
        s1 = self.e1(s0)
        s2 = self.e2(s1)
        s3 = self.e3(s2)
        b = self.bot(s3)
        u2 = self.u2(b, s2)
        u1 = self.u1(u2, s1)
        u0 = self.u0(u1, s0)
        return self.out(u0).squeeze(1)


def load_adapter_for_spec(spec):
    key = str(spec["ckpt_path"])
    if key in ADAPTER_CACHE:
        return ADAPTER_CACHE[key]
    ck = torch_load_any(spec["ckpt_path"], map_location="cpu")
    cfg = ck.get("run_config", {}) if isinstance(ck, dict) else {}
    model = MissingFrameAdapterUNet(
        in_ch=int(cfg.get("ADAPTER_IN_CH", 4)),
        base=int(cfg.get("ADAPTER_BASE", 48)),
        groups=int(cfg.get("ADAPTER_GROUPS", 8)),
        dropout=float(cfg.get("ADAPTER_DROPOUT", 0.03)),
    ).to(device)
    model.load_state_dict(ck["adapter"], strict=True)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    ADAPTER_CACHE[key] = (model, ck)
    print(f"Loaded residual-mined adapter: {spec['ckpt_path']}")
    return ADAPTER_CACHE[key]


def make_adapter_input_linear(mel_linear, mel_base, mask_mel):
    mask_bt = _mask_bt(mask_mel)
    m = _expand_mask_bt(mask_bt, mel_linear.shape[1])
    base_delta = mel_base - mel_linear
    return torch.stack([mel_linear, mel_base, base_delta, m], dim=1)


@torch.no_grad()
def run_residual_adapter_model(spec):
    # The residual adapter expects the earlier/retained mel estimate as context.
    fullskip_spec = next((s for s in resolved_specs if s.get("family") == "fullskip_plain2d" and s.get("found")), None)
    if fullskip_spec is None:
        raise RuntimeError("Residual adapter needs the full-skip 2D U-Net checkpoint to be found.")
    fullskip_model, _ = load_fullskip_for_spec(fullskip_spec)
    mel_linear = CLIP_CTX["mel_interp"]
    mask_mel = CLIP_CTX["mask_mel"]
    mask_bt = _mask_bt(mask_mel)
    delta_base = fullskip_model(mel_linear, mask_bt)
    mel_base = mel_linear + _expand_mask_bt(mask_bt, mel_linear.shape[1]) * delta_base

    adapter, ck = load_adapter_for_spec(spec)
    delta_v1 = adapter(make_adapter_input_linear(mel_linear, mel_base, mask_mel))
    alpha = float(spec.get("alpha", 1.0))
    mel_hat = mel_linear + _expand_mask_bt(mask_bt, mel_linear.shape[1]) * alpha * delta_v1
    wav_hat = mel_to_wave_bigvgan(mel_hat)[0]
    row = exact_summary_row(
        model_name=spec["name"],
        family=spec["family"],
        out_kind="BigVGAN",
        mel_hat=mel_hat,
        mel_real=CLIP_CTX["mel_real"],
        mask_mel=mask_mel,
        extra=dict(alpha=alpha, step=int(ck.get("step", -1)) if isinstance(ck, dict) else -1),
    )
    return dict(
        name=spec["name"],
        family=spec["family"],
        out_label=f'{spec["name"]} -> BigVGAN',
        audio=wav_hat.detach().cpu().numpy(),
        mel_hat=mel_hat[0].detach().cpu(),
        summary=row,
    )


In [ ]:
# =========================
# Per-family loading + exact-clip evaluation
# =========================

def _ckpt_config(ck):
    """Return whichever config dictionary a checkpoint uses."""
    rc = ck.get("run_config", None)
    if isinstance(rc, dict) and len(rc) > 0:
        return rc
    cfg = ck.get("config", None)
    if isinstance(cfg, dict):
        return cfg
    return {}


def _cfg_get(rc, *names, default=None):
    """Case-tolerant config getter for old and new notebooks."""
    for name in names:
        if name in rc:
            return rc[name]
    for name in names:
        upper = name.upper()
        if upper in rc:
            return rc[upper]
    for name in names:
        lower = name.lower()
        if lower in rc:
            return rc[lower]
    return default


def _get_stft_cfg_from_run_config(rc, default_n_fft, default_hop, default_win, default_center):
    stft_cfg = rc.get("stft", {})
    if not isinstance(stft_cfg, dict):
        stft_cfg = {}
    cstft_cfg = rc.get("complex_stft", {})
    if not isinstance(cstft_cfg, dict):
        cstft_cfg = {}

    n_fft = int(stft_cfg.get("n_fft", cstft_cfg.get("n_fft", _cfg_get(rc, "n_fft", "N_FFT", default=default_n_fft))))
    hop = int(stft_cfg.get("hop", cstft_cfg.get("hop", _cfg_get(rc, "hop", "HOP", default=default_hop))))
    win = int(stft_cfg.get("win", cstft_cfg.get("win", _cfg_get(rc, "win", "WIN", default=default_win))))
    center = bool(stft_cfg.get("center", cstft_cfg.get("center", _cfg_get(rc, "center", "CENTER", default=default_center))))
    return n_fft, hop, win, center

@torch.no_grad()
def run_plain2d_model(spec):
    ck = torch.load(str(spec["ckpt_path"]), map_location="cpu")
    rc = ck.get("run_config", {})
    frontend = ck.get("bigvgan_frontend_config", {})
    n_mels = int(frontend.get("num_mels", rc.get("n_mels", N_MELS)))
    base = int(rc.get("G_base", rc.get("g_base", G_BASE)))
    groups = int(rc.get("G_groups", rc.get("g_groups", G_GROUPS)))
    use_mask = bool(rc.get("G_use_mask", rc.get("g_use_mask", True)))

    G = PlainMelRefinerUNet2D(n_mels=n_mels, base=base, use_mask=use_mask, groups=groups).to(device)
    G.load_state_dict(ck["G"], strict=True)
    G.eval()

    mel_real = CLIP_CTX["mel_real"]
    mel_interp = CLIP_CTX["mel_interp"]
    mask_mel = CLIP_CTX["mask_mel"]

    delta = G(mel_interp, mask_mel[:, :, 0, :] if mask_mel.ndim == 4 else mask_mel)
    mask_apply = mask_mel[:, :, 0, :] if mask_mel.ndim == 4 else mask_mel
    mel_fixed = mel_interp + mask_apply * delta
    wav_fixed = mel_to_wave_bigvgan(mel_fixed)[0]

    row = exact_summary_row(
        model_name=spec["name"],
        family=spec["family"],
        out_kind="BigVGAN",
        mel_hat=mel_fixed,
        mel_real=mel_real,
        mask_mel=mask_mel,
        extra=dict(step=int(ck.get("step", -1)), variant=rc.get("experiment_variant", "")),
    )

    out = dict(
        name=spec["name"],
        family=spec["family"],
        out_label=f'{spec["name"]} -> BigVGAN',
        audio=wav_fixed.detach().cpu().numpy(),
        mel_hat=mel_fixed[0].detach().cpu(),
        summary=row,
    )

    del G, ck
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

@torch.no_grad()
def run_complexstft_model(spec):
    ck = torch.load(str(spec["ckpt_path"]), map_location="cpu")
    rc = _ckpt_config(ck)

    n_fft, hop, win, center = _get_stft_cfg_from_run_config(
        rc, default_n_fft=CPLX_N_FFT, default_hop=CPLX_HOP, default_win=CPLX_WIN, default_center=CPLX_CENTER
    )

    base = int(_cfg_get(rc, "g_base", "G_BASE", default=G_BASE))
    groups = int(_cfg_get(rc, "g_groups", "G_GROUPS", default=G_GROUPS))
    in_ch = int(_cfg_get(rc, "g_in_ch", "G_IN_CH", default=3))
    out_ch = int(_cfg_get(rc, "g_out_ch", "G_OUT_CH", default=2))

    G = ComplexSTFTUNet2D(in_ch=in_ch, out_ch=out_ch, base=base, groups=groups).to(device)
    state = ck.get("G", ck.get("model", ck.get("generator", ck.get("state_dict", None))))
    if state is None:
        raise KeyError(f"Could not find model state in {spec['ckpt_path']}. Expected key G/model/generator/state_dict.")
    G.load_state_dict(state, strict=True)
    G.eval()

    wb = CLIP_CTX["wav_batch"]
    X_real = stft_complex_cfg(wb, n_fft=n_fft, hop=hop, win=win, center=center)
    X_interp, mask_cplx = linear_time_inpaint_complex_cfg(X_real, k=K, offset=OFFSET)
    x_in = pack_complex_input(X_interp, mask_cplx)
    delta = G(x_in)
    dX = torch.complex(delta[:, 0], delta[:, 1])
    mask_ft = mask_cplx[:, 0].expand(-1, X_interp.shape[1], -1)
    X_hat = X_interp + dX * mask_ft
    wav_hat = istft_complex_cfg(X_hat, n_fft=n_fft, hop=hop, win=win, center=center, length=CLIP_CTX["wav_real"].shape[-1])
    wav_hat = wav_hat.clamp(-1, 1)
    mel_hat = wav_to_bigvgan_mel(wav_hat)

    mel_hat, mel_real, mask_mel = trim_to_common_length(mel_hat, CLIP_CTX["mel_real"], CLIP_CTX["mask_mel"])
    row = exact_summary_row(
        model_name=spec["name"],
        family=spec["family"],
        out_kind=f"ISTFT center={center}",
        mel_hat=mel_hat,
        mel_real=mel_real,
        mask_mel=mask_mel,
        extra=dict(
            step=int(ck.get("step", -1)),
            stft_center=center,
            n_fft=n_fft,
            hop=hop,
            win=win,
        ),
    )

    out = dict(
        name=spec["name"],
        family=spec["family"],
        out_label=f'{spec["name"]} -> ISTFT',
        audio=wav_hat[0].detach().cpu().numpy(),
        mel_hat=mel_hat[0].detach().cpu(),
        summary=row,
    )

    del G, ck
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

@torch.no_grad()
def run_hybrid_model(spec):
    ck = torch.load(str(spec["ckpt_path"]), map_location="cpu")
    rc = ck.get("run_config", {})

    sr = int(rc.get("sr", SR))
    assert sr == SR, f"Hybrid checkpoint SR={sr} differs from notebook SR={SR}; adjust config first if needed."

    n_fft = int(rc.get("mel", {}).get("n_fft", N_FFT)) if isinstance(rc.get("mel", None), dict) else N_FFT
    hop = int(rc.get("mel", {}).get("hop", HOP)) if isinstance(rc.get("mel", None), dict) else HOP
    win = int(rc.get("mel", {}).get("win", WIN)) if isinstance(rc.get("mel", None), dict) else WIN
    n_mels = int(rc.get("mel", {}).get("n_mels", N_MELS)) if isinstance(rc.get("mel", None), dict) else N_MELS
    c_n_fft = int(rc.get("complex_stft", {}).get("n_fft", CPLX_N_FFT)) if isinstance(rc.get("complex_stft", None), dict) else CPLX_N_FFT
    c_hop = int(rc.get("complex_stft", {}).get("hop", CPLX_HOP)) if isinstance(rc.get("complex_stft", None), dict) else CPLX_HOP
    c_win = int(rc.get("complex_stft", {}).get("win", CPLX_WIN)) if isinstance(rc.get("complex_stft", None), dict) else CPLX_WIN
    c_center = bool(rc.get("complex_stft", {}).get("center", CPLX_CENTER)) if isinstance(rc.get("complex_stft", None), dict) else CPLX_CENTER

    g_base = int(rc.get("g_base", G_BASE))
    cplx_base = int(rc.get("cplx_base", CPLX_BASE))
    fusion_base = int(rc.get("fusion_base", FUSION_BASE))
    groups = int(rc.get("g_groups", G_GROUPS)) if "g_groups" in rc else G_GROUPS

    MelG = MelBranchUNet2D(n_mels=n_mels, base=g_base, use_mask=True, groups=groups).to(device)
    CplxG = ComplexBranchUNet2D(in_ch=3, out_ch=2, base=cplx_base, groups=groups).to(device)
    prev_gate_temp = globals().get("GATE_TEMP", 2.0)
    globals()["GATE_TEMP"] = float(rc.get("gate_temp", prev_gate_temp))
    FuseG = FusionUNet2D(in_ch=5, base=fusion_base, groups=groups).to(device)

    MelG.load_state_dict(ck["MelG"], strict=True)
    CplxG.load_state_dict(ck["CplxG"], strict=True)
    FuseG.load_state_dict(ck["FuseG"], strict=True)

    MelG.eval(); CplxG.eval(); FuseG.eval()

    wb = CLIP_CTX["wav_batch"]
    mel_real = CLIP_CTX["mel_real"]
    mel_interp = CLIP_CTX["mel_interp"]
    mask_mel = CLIP_CTX["mask_mel"]
    X_real = stft_complex_cfg(wb, n_fft=c_n_fft, hop=c_hop, win=c_win, center=c_center)
    X_interp, mask_cplx = linear_time_inpaint_complex_cfg(X_real, k=K, offset=OFFSET)

    delta_mel = MelG(mel_interp, mask_mel[:, :, 0, :])
    mel_main_hat = mel_interp + delta_mel * mask_mel[:, 0].expand(-1, mel_interp.shape[1], -1)

    x_in = pack_complex_input(X_interp, mask_cplx)
    delta_c = CplxG(x_in)
    dX = torch.complex(delta_c[:, 0], delta_c[:, 1])
    mask_ft = mask_cplx[:, 0].expand(-1, X_interp.shape[1], -1)
    X_hat = X_interp + dX * mask_ft
    wav_cplx_hat = istft_complex_cfg(X_hat, n_fft=c_n_fft, hop=c_hop, win=c_win, center=c_center, length=CLIP_CTX["wav_real"].shape[-1])
    mel_cplx_hat = wav_to_bigvgan_mel(wav_cplx_hat)

    mel_real, mel_interp, mel_main_hat, mel_cplx_hat, mask_mel = trim_to_common_length(
        mel_real, mel_interp, mel_main_hat, mel_cplx_hat, mask_mel
    )

    prior = make_hf_prior(mask_mel, n_mels=mel_real.shape[1], radius=PRIOR_RADIUS)
    gate, delta_fuse = FuseG(mel_interp, mel_main_hat, mel_cplx_hat, prior, mask_mel[:, :, 0, :])
    mel_fused = mel_main_hat + prior * (gate * (mel_cplx_hat - mel_main_hat) + delta_fuse)

    wav_fused = mel_to_wave_bigvgan(mel_fused)[0]

    row = exact_summary_row(
        model_name=spec["name"],
        family=spec["family"],
        out_kind="BigVGAN",
        mel_hat=mel_fused,
        mel_real=mel_real,
        mask_mel=mask_mel,
        extra=dict(step=int(ck.get("step", -1)), avg_gate=float(gate.mean().item())),
    )

    out = dict(
        name=spec["name"],
        family=spec["family"],
        out_label=f'{spec["name"]} -> BigVGAN',
        audio=wav_fused.detach().cpu().numpy(),
        mel_hat=mel_fused[0].detach().cpu(),
        mel_main=mel_main_hat[0].detach().cpu(),
        mel_cplx=mel_cplx_hat[0].detach().cpu(),
        summary=row,
    )

    globals()["GATE_TEMP"] = prev_gate_temp
    del MelG, CplxG, FuseG, ck
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

@torch.no_grad()
def run_meldom_stft_assist_model(spec):
    ck = torch.load(str(spec["ckpt_path"]), map_location="cpu")
    rc = ck.get("run_config", {})

    mel_cfg = rc.get("mel", {}) if isinstance(rc.get("mel", None), dict) else {}
    cplx_cfg = rc.get("complex_stft", {}) if isinstance(rc.get("complex_stft", None), dict) else {}
    n_mels = int(mel_cfg.get("n_mels", N_MELS))
    c_n_fft = int(cplx_cfg.get("n_fft", CPLX_N_FFT))
    c_hop = int(cplx_cfg.get("hop", CPLX_HOP))
    c_win = int(cplx_cfg.get("win", CPLX_WIN))
    c_center = bool(cplx_cfg.get("center", True))

    groups = int(rc.get("g_groups", G_GROUPS))
    g_base = int(rc.get("g_base", G_BASE))
    gate_temp_local = float(rc.get("gate_temp", rc.get("GATE_TEMP", 2.2)))

    mel_teacher_ckpt = ck.get("mel_teacher_ckpt", None)
    complex_teacher_ckpt = ck.get("complex_teacher_ckpt", None)
    if mel_teacher_ckpt is None:
        mel_teacher_ckpt = rc.get("teachers", {}).get("mel_teacher_ckpt", None) if isinstance(rc.get("teachers", None), dict) else None
    if complex_teacher_ckpt is None:
        complex_teacher_ckpt = rc.get("teachers", {}).get("complex_teacher_ckpt", None) if isinstance(rc.get("teachers", None), dict) else None
    assert mel_teacher_ckpt is not None and Path(mel_teacher_ckpt).exists(), "Missing mel teacher checkpoint for mel-dominant STFT-assist model."
    assert complex_teacher_ckpt is not None and Path(complex_teacher_ckpt).exists(), "Missing complex teacher checkpoint for mel-dominant STFT-assist model."

    # load frozen teachers
    ck_mel = torch.load(str(mel_teacher_ckpt), map_location="cpu")
    rc_mel = ck_mel.get("run_config", {})
    mel_base = int(rc_mel.get("g_base", rc_mel.get("G_base", 24)))
    mel_groups = int(rc_mel.get("g_groups", rc_mel.get("G_groups", 8)))
    mel_use_mask = bool(rc_mel.get("G_use_mask", True)) if "G_use_mask" in rc_mel else True

    MelTeacher = MelBranchUNet2D(n_mels=n_mels, base=mel_base, use_mask=mel_use_mask, groups=mel_groups).to(device)
    MelTeacher.load_state_dict(ck_mel["G"], strict=True)
    MelTeacher.eval()

    ck_cplx = torch.load(str(complex_teacher_ckpt), map_location="cpu")
    rc_cplx = ck_cplx.get("run_config", {})
    cplx_base = int(rc_cplx.get("g_base", rc_cplx.get("G_base", 24)))
    cplx_groups = int(rc_cplx.get("g_groups", rc_cplx.get("G_GROUPS", 8)))
    cplx_in = int(rc_cplx.get("g_in_ch", rc_cplx.get("G_IN_CH", 3)))
    cplx_out = int(rc_cplx.get("g_out_ch", rc_cplx.get("G_OUT_CH", 2)))

    ComplexTeacher = ComplexBranchUNet2D(in_ch=cplx_in, out_ch=cplx_out, base=cplx_base, groups=cplx_groups).to(device)
    ComplexTeacher.load_state_dict(ck_cplx["G"], strict=True)
    ComplexTeacher.eval()

    prev_gate_temp = globals().get("GATE_TEMP", 2.0)
    globals()["GATE_TEMP"] = gate_temp_local
    FuseG = FusionUNet2D(in_ch=5, base=g_base, groups=groups).to(device)
    FuseG.load_state_dict(ck["G"], strict=True)
    FuseG.eval()

    wb = CLIP_CTX["wav_batch"]
    mel_real = CLIP_CTX["mel_real"]
    mel_interp = CLIP_CTX["mel_interp"]
    mask_mel = CLIP_CTX["mask_mel"]

    X_real = stft_complex_cfg(wb, n_fft=c_n_fft, hop=c_hop, win=c_win, center=c_center)
    X_interp, mask_cplx = linear_time_inpaint_complex_cfg(X_real, k=K, offset=OFFSET)

    mel_teacher_hat = MelTeacher(mel_interp, mask_mel[:, :, 0, :])
    mel_teacher_hat = mel_interp + mel_teacher_hat * mask_mel[:, 0].expand(-1, mel_interp.shape[1], -1)

    x_in = pack_complex_input(X_interp, mask_cplx)
    delta_c = ComplexTeacher(x_in)
    dX = torch.complex(delta_c[:, 0], delta_c[:, 1])
    mask_ft = mask_cplx[:, 0].expand(-1, X_interp.shape[1], -1)
    X_hat = X_interp + dX * mask_ft
    wav_cplx_hat = istft_complex_cfg(X_hat, n_fft=c_n_fft, hop=c_hop, win=c_win, center=c_center, length=CLIP_CTX["wav_real"].shape[-1])
    mel_cplx_hat = wav_to_bigvgan_mel(wav_cplx_hat)

    mel_real, mel_interp, mel_teacher_hat, mel_cplx_hat, mask_mel = trim_to_common_length(
        mel_real, mel_interp, mel_teacher_hat, mel_cplx_hat, mask_mel
    )

    prior_radius = int(rc.get("prior_radius", rc.get("PRIOR_RADIUS", PRIOR_RADIUS)))
    prior_boundary_gain = float(rc.get("prior_boundary_gain", rc.get("PRIOR_BOUNDARY_GAIN", PRIOR_BOUNDARY_GAIN)))
    prior_hf_power = float(rc.get("prior_hf_power", rc.get("PRIOR_HF_POWER", PRIOR_HF_POWER)))
    prior = make_hf_prior(mask_mel, n_mels=mel_real.shape[1], radius=prior_radius,
                          boundary_gain=prior_boundary_gain, hf_power=prior_hf_power)

    gate, delta_fuse = FuseG(mel_interp, mel_teacher_hat, mel_cplx_hat, prior, mask_mel[:, :, 0, :])
    mel_fused = mel_teacher_hat + prior * (gate * (mel_cplx_hat - mel_teacher_hat) + delta_fuse)
    wav_fused = mel_to_wave_bigvgan(mel_fused)[0]

    row = exact_summary_row(
        model_name=spec["name"],
        family=spec["family"],
        out_kind="BigVGAN",
        mel_hat=mel_fused,
        mel_real=mel_real,
        mask_mel=mask_mel,
        extra=dict(step=int(ck.get("step", -1)), avg_gate=float(gate.mean().item())),
    )

    out = dict(
        name=spec["name"],
        family=spec["family"],
        out_label=f'{spec["name"]} -> BigVGAN',
        audio=wav_fused.detach().cpu().numpy(),
        mel_hat=mel_fused[0].detach().cpu(),
        mel_main=mel_teacher_hat[0].detach().cpu(),
        mel_cplx=mel_cplx_hat[0].detach().cpu(),
        summary=row,
    )

    globals()["GATE_TEMP"] = prev_gate_temp
    del MelTeacher, ComplexTeacher, FuseG, ck, ck_mel, ck_cplx
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out


@torch.no_grad()
def run_complexmask_model(spec):
    """
    Evaluate complex-mask mel/STFT fusion checkpoints.

    Listening output is direct ISTFT from the blended complex STFT.
    Metrics are computed by converting that waveform back to BigVGAN mel so
    it can be compared against the other mel-domain models on the same table.
    """
    ck = torch.load(str(spec["ckpt_path"]), map_location="cpu")
    rc = ck.get("run_config", {})
    arch = rc.get("arch", {}) if isinstance(rc.get("arch", None), dict) else {}
    losses_cfg = rc.get("losses", {}) if isinstance(rc.get("losses", None), dict) else {}

    # Model architecture from run_config, with safe defaults.
    g_base = int(arch.get("g_base", rc.get("g_base", G_BASE)))
    g_groups = int(arch.get("g_groups", rc.get("g_groups", G_GROUPS)))
    fusion_in_ch = int(arch.get("fusion_in_ch", rc.get("fusion_in_ch", FUSION_IN_CH)))
    mask_temp = float(arch.get("mask_temp", rc.get("mask_temp", MASK_TEMP)))
    mask_real_max = float(arch.get("mask_real_max", rc.get("mask_real_max", MASK_REAL_MAX)))
    mask_imag_scale = float(arch.get("mask_imag_scale", rc.get("mask_imag_scale", MASK_IMAG_SCALE)))
    mask_mode = str(arch.get("mask_mode", rc.get("mask_mode", "complex")))

    Gcm = ComplexMaskFusionUNet2D(
        in_ch=fusion_in_ch,
        base=g_base,
        groups=g_groups,
        mask_temp=mask_temp,
        mask_real_max=mask_real_max,
        mask_imag_scale=mask_imag_scale,
        mask_mode=mask_mode,
    ).to(device)
    Gcm.load_state_dict(ck["G"], strict=True)
    Gcm.eval()

    # Resolve teacher checkpoints saved inside the fusion checkpoint.
    mel_teacher_ckpt = ck.get("mel_teacher_ckpt", None)
    complex_teacher_ckpt = ck.get("complex_teacher_ckpt", None)
    if mel_teacher_ckpt is None and isinstance(rc.get("teachers", None), dict):
        mel_teacher_ckpt = rc["teachers"].get("mel_teacher_ckpt", None)
    if complex_teacher_ckpt is None and isinstance(rc.get("teachers", None), dict):
        complex_teacher_ckpt = rc["teachers"].get("complex_teacher_ckpt", None)

    assert mel_teacher_ckpt is not None and Path(mel_teacher_ckpt).exists(), (
        f"Missing mel teacher checkpoint for {spec['name']}: {mel_teacher_ckpt}"
    )
    assert complex_teacher_ckpt is not None and Path(complex_teacher_ckpt).exists(), (
        f"Missing complex teacher checkpoint for {spec['name']}: {complex_teacher_ckpt}"
    )

    # Load mel teacher.
    ck_mel = torch.load(str(mel_teacher_ckpt), map_location="cpu")
    rc_mel = ck_mel.get("run_config", {}) if isinstance(ck_mel, dict) else {}
    sd_mel = extract_state_dict(ck_mel)
    mel_base = int(cfg_get(rc_mel, ["g_base", "G_BASE", "G_base", "base", ("arch", "g_base")], infer_base_from_state_dict(sd_mel, 24)))
    mel_groups = int(cfg_get(rc_mel, ["g_groups", "G_GROUPS", "G_groups", "groups", ("arch", "g_groups")], G_GROUPS))
    mel_use_mask = bool(cfg_get(rc_mel, ["g_use_mask", "G_USE_MASK", "G_use_mask", "use_mask", ("arch", "g_use_mask")], True))
    mel_n_mels = int(cfg_get(rc_mel, ["n_mels", "N_MELS", ("mel", "n_mels")], N_MELS))
    MelTeacher = MelBranchUNet2D(n_mels=mel_n_mels, base=mel_base, use_mask=mel_use_mask, groups=mel_groups).to(device)
    MelTeacher.load_state_dict(sd_mel, strict=True)
    MelTeacher.eval()

    # Load complex STFT teacher.
    ck_cplx = torch.load(str(complex_teacher_ckpt), map_location="cpu")
    rc_cplx = ck_cplx.get("run_config", {}) if isinstance(ck_cplx, dict) else {}
    sd_cplx = extract_state_dict(ck_cplx)
    cplx_base = int(cfg_get(rc_cplx, ["g_base", "G_BASE", "G_base", "base", ("arch", "g_base")], infer_base_from_state_dict(sd_cplx, 24)))
    cplx_groups = int(cfg_get(rc_cplx, ["g_groups", "G_GROUPS", "G_groups", "groups", ("arch", "g_groups")], G_GROUPS))
    cplx_in = int(cfg_get(rc_cplx, ["g_in_ch", "G_IN_CH", ("arch", "g_in_ch")], 3))
    cplx_out = int(cfg_get(rc_cplx, ["g_out_ch", "G_OUT_CH", ("arch", "g_out_ch")], 2))
    ComplexTeacher = ComplexBranchUNet2D(in_ch=cplx_in, out_ch=cplx_out, base=cplx_base, groups=cplx_groups).to(device)
    ComplexTeacher.load_state_dict(sd_cplx, strict=True)
    ComplexTeacher.eval()

    # STFT configs.
    cplx_cfg = rc.get("complex_teacher_stft", {}) if isinstance(rc.get("complex_teacher_stft", None), dict) else {}
    blend_cfg = rc.get("blend_stft", {}) if isinstance(rc.get("blend_stft", None), dict) else {}
    c_n_fft = int(cplx_cfg.get("n_fft", CPLX_N_FFT))
    c_hop = int(cplx_cfg.get("hop", CPLX_HOP))
    c_win = int(cplx_cfg.get("win", CPLX_WIN))
    c_center = bool(cplx_cfg.get("center", True))

    b_n_fft = int(blend_cfg.get("n_fft", BLEND_N_FFT))
    b_hop = int(blend_cfg.get("hop", BLEND_HOP))
    b_win = int(blend_cfg.get("win", BLEND_WIN))
    b_center = bool(blend_cfg.get("center", BLEND_CENTER))

    stft_hf_start = float(losses_cfg.get("stft_hf_start_frac", HF_START_FRAC))
    stft_hf_end = float(losses_cfg.get("stft_hf_end_gain", 6.0))
    stft_hf_power = float(losses_cfg.get("stft_hf_ramp_power", HF_RAMP_POWER))
    prior_radius = int(losses_cfg.get("prior_radius", PRIOR_RADIUS))
    prior_boundary_gain = float(losses_cfg.get("prior_boundary_gain", PRIOR_BOUNDARY_GAIN))
    prior_hf_power = float(losses_cfg.get("prior_hf_power", PRIOR_HF_POWER))
    mask_dilate_local = int(losses_cfg.get("mask_dilate", MASK_DILATE))

    stft_freq_weights = build_freq_weights(b_n_fft // 2 + 1, stft_hf_start, stft_hf_end, stft_hf_power, device=device)

    wb = CLIP_CTX["wav_batch"]
    target_len = CLIP_CTX["wav_real"].shape[-1]
    mel_real = CLIP_CTX["mel_real"]
    mel_interp = CLIP_CTX["mel_interp"]
    mask_mel = CLIP_CTX["mask_mel"]

    # Mel teacher route -> waveform -> blend STFT grid.
    delta_mel = MelTeacher(mel_interp, mask_mel[:, :, 0, :])
    mel_teacher_hat = mel_interp + delta_mel * mask_mel[:, 0]
    wav_mel = match_wav_length(mel_to_wave_bigvgan(mel_teacher_hat), target_len)

    # Complex teacher route -> waveform -> blend STFT grid.
    X_teacher_real = stft_complex_cfg(wb, n_fft=c_n_fft, hop=c_hop, win=c_win, center=c_center)
    X_interp_teacher, mask_cplx = linear_time_inpaint_complex_cfg(X_teacher_real, k=K, offset=OFFSET)
    x_in = pack_complex_input(X_interp_teacher, mask_cplx)
    delta_c = ComplexTeacher(x_in)
    dX = torch.complex(delta_c[:, 0], delta_c[:, 1])
    mask_ft = mask_cplx[:, 0].expand(-1, X_interp_teacher.shape[1], -1)
    X_teacher_hat = X_interp_teacher + dX * mask_ft
    wav_stft = istft_complex_cfg(X_teacher_hat, n_fft=c_n_fft, hop=c_hop, win=c_win, center=c_center, length=target_len)
    wav_stft = match_wav_length(wav_stft, target_len)

    # Complex-mask blend on shared STFT grid.
    X_real = stft_complex_cfg(wb, n_fft=b_n_fft, hop=b_hop, win=b_win, center=b_center)
    X_mel = stft_complex_cfg(wav_mel, n_fft=b_n_fft, hop=b_hop, win=b_win, center=b_center)
    X_stft = stft_complex_cfg(wav_stft, n_fft=b_n_fft, hop=b_hop, win=b_win, center=b_center)
    X_real, X_mel, X_stft = trim_complex_to_common(X_real, X_mel, X_stft)

    mask_blend = match_mask_frames(mask_cplx, X_real.shape[-1])
    prior = make_stft_prior_eval(
        mask_blend, X_real.shape[1], stft_freq_weights,
        radius=prior_radius, boundary_gain=prior_boundary_gain,
        hf_power=prior_hf_power, stft_hf_end_gain=stft_hf_end,
    )
    prior = match_frames_tensor(prior, X_real.shape[-1])

    feat, prior, _ = build_complexmask_fusion_features(X_mel, X_stft, prior, mask_blend, fusion_in_ch=fusion_in_ch)
    C, alpha, beta = Gcm(feat)
    X_blend = blend_with_complex_mask_eval(X_mel, X_stft, C, prior)
    wav_blend = istft_complex_cfg(X_blend, n_fft=b_n_fft, hop=b_hop, win=b_win, center=b_center, length=target_len)
    wav_blend = wav_blend.clamp(-1.0, 1.0)

    # Convert blended waveform to mel for common table metrics.
    mel_blend = wav_to_bigvgan_mel(wav_blend)
    mel_blend, mel_real_t, mask_mel_t = trim_to_common_length(mel_blend, mel_real, mask_mel)

    stft_loghf_mel = masked_logmag_l1(X_mel, X_real, mask_blend, freq_weights=stft_freq_weights, mask_dilate=mask_dilate_local).item()
    stft_loghf_stft = masked_logmag_l1(X_stft, X_real, mask_blend, freq_weights=stft_freq_weights, mask_dilate=mask_dilate_local).item()
    stft_loghf_blend = masked_logmag_l1(X_blend, X_real, mask_blend, freq_weights=stft_freq_weights, mask_dilate=mask_dilate_local).item()
    alpha_mean = float((alpha.detach() * prior).sum() / (prior.sum() + 1e-8))
    beta_abs_mean = float((beta.detach().abs() * prior).sum() / (prior.sum() + 1e-8))

    row = exact_summary_row(
        model_name=spec["name"],
        family=spec["family"],
        out_kind="Complex-mask ISTFT",
        mel_hat=mel_blend,
        mel_real=mel_real_t,
        mask_mel=mask_mel_t,
        extra=dict(
            step=int(ck.get("step", -1)),
            mask_mode=mask_mode,
            stft_loghf_melroute=stft_loghf_mel,
            stft_loghf_stftroute=stft_loghf_stft,
            stft_loghf_blend=stft_loghf_blend,
            alpha_mean=alpha_mean,
            beta_abs_mean=beta_abs_mean,
            mel_teacher=str(mel_teacher_ckpt),
            complex_teacher=str(complex_teacher_ckpt),
        ),
    )

    out = dict(
        name=spec["name"],
        family=spec["family"],
        out_label=f'{spec["name"]} -> complex-mask ISTFT',
        audio=wav_blend[0].detach().cpu().numpy(),
        mel_hat=mel_blend[0].detach().cpu(),
        summary=row,
        wav_mel=wav_mel[0].detach().cpu().numpy(),
        wav_stft=wav_stft[0].detach().cpu().numpy(),
    )

    del Gcm, MelTeacher, ComplexTeacher, ck, ck_mel, ck_cplx
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out



@torch.no_grad()
def run_complexstft_crn_model(spec):
    """Evaluate the DCCRN-lite / CRN U-Net checkpoint on the exact 6s clip."""
    ck = torch.load(str(spec["ckpt_path"]), map_location="cpu")
    rc = _ckpt_config(ck)

    n_fft = int(rc.get("N_FFT", CPLX_N_FFT))
    hop = int(rc.get("HOP", CPLX_HOP))
    win = int(rc.get("WIN", CPLX_WIN))
    center = bool(rc.get("CENTER", True))
    base = int(rc.get("G_BASE", 16))
    groups = int(rc.get("G_GROUPS", G_GROUPS))
    in_ch = int(rc.get("G_IN_CH", 3))
    out_ch = int(rc.get("G_OUT_CH", 2))
    hidden = int(rc.get("CRN_HIDDEN", 256))
    layers = int(rc.get("CRN_LAYERS", 1))
    dropout = float(rc.get("CRN_DROPOUT", 0.10))
    bot_bins = int(rc.get("BOT_FREQ_BINS", (n_fft // 2 + 1) // 8))

    Gcrn = ComplexSTFTCRNUNet2D(
        in_ch=in_ch,
        out_ch=out_ch,
        base=base,
        groups=groups,
        bottleneck_freq_bins=bot_bins,
        crn_hidden=hidden,
        crn_layers=layers,
        crn_dropout=dropout,
    ).to(device)
    state = extract_state_dict(ck)
    Gcrn.load_state_dict(state, strict=True)
    Gcrn.eval()

    wb = CLIP_CTX["wav_batch"]
    X_real = stft_complex_cfg(wb, n_fft=n_fft, hop=hop, win=win, center=center)
    X_interp, mask_cplx = linear_time_inpaint_complex_cfg(X_real, k=K, offset=OFFSET)
    x_in = pack_complex_input(X_interp, mask_cplx)
    delta = Gcrn(x_in)
    dX = torch.complex(delta[:, 0], delta[:, 1])
    mask_ft = mask_cplx[:, 0].expand(-1, X_interp.shape[1], -1)
    X_hat = X_interp + dX * mask_ft
    wav_hat = istft_complex_cfg(X_hat, n_fft=n_fft, hop=hop, win=win, center=center, length=CLIP_CTX["wav_real"].shape[-1])
    wav_hat = wav_hat.clamp(-1, 1)
    mel_hat = wav_to_bigvgan_mel(wav_hat)

    mel_hat, mel_real, mask_mel = trim_to_common_length(mel_hat, CLIP_CTX["mel_real"], CLIP_CTX["mask_mel"])
    row = exact_summary_row(
        model_name=spec["name"],
        family=spec["family"],
        out_kind=f"ISTFT center={center}",
        mel_hat=mel_hat,
        mel_real=mel_real,
        mask_mel=mask_mel,
        extra=dict(
            step=int(ck.get("step", -1)),
            stft_center=center,
            n_fft=n_fft,
            hop=hop,
            win=win,
            crn_hidden=hidden,
        ),
    )

    out = dict(
        name=spec["name"],
        family=spec["family"],
        out_label=f'{spec["name"]} -> ISTFT',
        audio=wav_hat[0].detach().cpu().numpy(),
        mel_hat=mel_hat[0].detach().cpu(),
        summary=row,
    )

    del Gcrn, ck
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out


def run_model(spec):
    if spec["family"] == "fullskip_plain2d":
        return run_fullskip_model(spec)
    if spec["family"] == "residual_adapter":
        return run_residual_adapter_model(spec)
    if spec["family"] == "plain2d":
        return run_plain2d_model(spec)
    if spec["family"] == "complexstft":
        return run_complexstft_model(spec)
    if spec["family"] == "hybrid":
        return run_hybrid_model(spec)
    if spec["family"] == "meldom_stft_assist":
        return run_meldom_stft_assist_model(spec)
    if spec["family"] == "complexmask":
        return run_complexmask_model(spec)
    if spec["family"] == "complexstft_crn":
        return run_complexstft_crn_model(spec)
    raise ValueError(f"Unknown family: {spec['family']}")


In [ ]:
# =========================
# Configurable listening helpers: inpainting + actual time stretching
# =========================
SPEECH_LISTEN_SEG_S = float(LISTENING_SEG_S_BY_DOMAIN.get("speech", 6.0))
MUSIC_LISTEN_SEG_S = float(LISTENING_SEG_S_BY_DOMAIN.get("music", 10.0))

def _listening_split_for_domain(domain: str):
    domain = str(domain).lower().strip()
    if domain not in {"speech", "music"}:
        raise ValueError("LISTENING_DOMAIN must be 'speech' or 'music'")
    return f"{domain}_test.txt"

def build_listening_cases():
    split = _listening_split_for_domain(LISTENING_DOMAIN)
    source_type = LISTENING_DOMAIN.lower().strip()
    seg_s = float(LISTENING_SEG_S_BY_DOMAIN.get(source_type, 6.0))
    start_s = float(LISTENING_START_SEC)
    cases = []
    if RUN_LISTENING_EVAL and LISTENING_INCLUDE_1X_INPAINT:
        cases.append(dict(
            label=f"{source_type.upper()} idx{LISTENING_INDEX} 1x inpainting",
            split=split,
            index=int(LISTENING_INDEX),
            source_type=source_type,
            mode="inpaint",
            k=1,
            offset=0,
            seg_s=seg_s,
            start_s=start_s,
        ))
    if RUN_LISTENING_EVAL:
        for speed in LISTENING_STRETCH_SPEEDS:
            cases.append(dict(
                label=f"{source_type.upper()} idx{LISTENING_INDEX} stretch {float(speed):.2f}x",
                split=split,
                index=int(LISTENING_INDEX),
                source_type=source_type,
                mode="stretch",
                speed=float(speed),
                seg_s=seg_s,
                start_s=start_s,
            ))
    return cases

EXPANDED_EVAL_CASES = build_listening_cases()

# Save references and model outputs as .wav files as well as displaying them.
EVAL_OUT_DIR = RUN_DIR / "listening_examples"
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)


def resolve_case_path(split_name: str, index: int):
    p = SPLIT_DIR / split_name
    assert p.exists(), f"Missing split file: {p}"
    lines = [ln.strip() for ln in p.read_text().splitlines() if ln.strip()]
    assert 0 <= index < len(lines), f"Index {index} out of range for {split_name} with {len(lines)} files"
    return Path(lines[index])


def load_wav_segment(path: Path, seg_s: float, start_s: float = 0.0):
    wav, sr = sf.read(str(path), always_2d=False)
    wav = np.asarray(wav)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    wav = torch.tensor(wav, dtype=torch.float32)
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    start = int(start_s * SR)
    need = int(seg_s * SR)
    if wav.numel() < start + need:
        wav = F.pad(wav, (0, max(0, start + need - wav.numel())))
    wav = wav[start:start+need]
    peak = wav.abs().max().clamp_min(1e-8)
    if peak > 1.0:
        wav = wav / peak
    return wav.to(device)


def linear_resample_time_real(x_bft: torch.Tensor, speed: float):
    """Linear time-axis resampling for real tensors [B,F,T]. Returns [B,F,T_out] and mask [B,1,F,T_out]."""
    assert x_bft.ndim == 3, tuple(x_bft.shape)
    B, Freq, T = x_bft.shape
    if T <= 1:
        mask = torch.zeros(B, 1, Freq, T, device=x_bft.device, dtype=x_bft.dtype)
        return x_bft, mask
    speed = float(speed)
    assert speed > 0, speed
    T_out = int(round((T - 1) / speed)) + 1
    pos = torch.arange(T_out, device=x_bft.device, dtype=torch.float32) * speed
    pos = pos.clamp(0, T - 1)
    lo = torch.floor(pos).long()
    hi = torch.ceil(pos).long()
    a = (pos - lo.float()).view(1, 1, T_out)
    out = (1.0 - a) * x_bft[:, :, lo] + a * x_bft[:, :, hi]
    inserted = (hi != lo).float().view(1, 1, 1, T_out)
    mask = inserted.expand(B, 1, Freq, T_out).to(dtype=x_bft.dtype)
    return out, mask


def linear_resample_time_complex(X_bft: torch.Tensor, speed: float):
    """Linear time-axis resampling for complex tensors [B,F,T]. Returns complex [B,F,T_out] and mask [B,1,F,T_out]."""
    real, mask = linear_resample_time_real(X_bft.real, speed)
    imag, _ = linear_resample_time_real(X_bft.imag, speed)
    return torch.complex(real, imag), mask


_ORIGINAL_linear_time_inpaint_complex_cfg = linear_time_inpaint_complex_cfg

def linear_time_inpaint_complex_cfg(X: torch.Tensor, k: int, offset: int):
    """For normal cases, use k/offset inpainting. For stretch cases, ignore k/offset and resample frames by speed."""
    if isinstance(globals().get("CLIP_CTX", None), dict) and CLIP_CTX.get("mode") == "stretch":
        return linear_resample_time_complex(X, CLIP_CTX["speed"])
    return _ORIGINAL_linear_time_inpaint_complex_cfg(X, k=k, offset=offset)


def build_case_context(case: dict):
    path = resolve_case_path(case["split"], case["index"])
    wav_orig = load_wav_segment(path, seg_s=float(case.get("seg_s", SEG_S)), start_s=float(case.get("start_s", 0.0)))
    wav_batch = wav_orig.unsqueeze(0)
    mel_src = wav_to_bigvgan_mel(wav_batch)

    mode = case["mode"]
    if mode == "inpaint":
        k = int(case.get("k", K))
        offset = int(case.get("offset", OFFSET))
        mel_interp, mask_mel = linear_time_inpaint_mel(mel_src, k=k, offset=offset)
        target_len = wav_orig.shape[-1]
        mel_ref = mel_src
        wav_len_ref = wav_orig
        has_ground_truth = True
        display_subtitle = f"split={case['split']} | index={case['index']} | k={k} | offset={offset} | {float(case.get('seg_s', SEG_S)):.1f}s"
    elif mode == "stretch":
        speed = float(case["speed"])
        mel_interp, mask_mel = linear_resample_time_real(mel_src, speed=speed)
        target_len = int(round((wav_orig.shape[-1] - 1) / speed)) + 1
        mel_ref = mel_interp
        wav_len_ref = torch.zeros(target_len, device=device)
        has_ground_truth = False
        display_subtitle = f"split={case['split']} | index={case['index']} | speed={speed:.2f}x | input={float(case.get('seg_s', SEG_S)):.1f}s | output~{target_len/SR:.2f}s"
    else:
        raise ValueError(f"Unknown case mode: {mode}")

    return dict(
        mode=mode,
        speed=float(case.get("speed", 1.0)),
        path=path,
        wav_original=wav_orig,
        wav_batch=wav_batch,
        wav_real=wav_len_ref,
        mel_real=mel_ref,
        mel_ref=mel_ref,
        mel_interp=mel_interp,
        mask_mel=mask_mel,
        target_len=target_len,
        has_ground_truth=has_ground_truth,
        display_subtitle=display_subtitle,
    )


def safe_filename(s: str):
    return "".join(c if c.isalnum() or c in "-_." else "_" for c in s)[:120]


def save_audio_if_requested(case_label, name, audio_np, sr=SR):
    if not SAVE_AUDIO_FILES:
        return None
    case_dir = EVAL_OUT_DIR / safe_filename(case_label)
    case_dir.mkdir(parents=True, exist_ok=True)
    p = case_dir / f"{safe_filename(name)}.wav"
    arr = np.asarray(audio_np, dtype=np.float32)
    arr = np.nan_to_num(arr)
    mx = np.max(np.abs(arr)) if arr.size else 0.0
    if mx > 1.0:
        arr = arr / mx
    sf.write(str(p), arr, sr)
    return p


def no_ref_metrics(name: str, family: str, audio_np, mel_hat_cpu, ctx):
    """Proxy diagnostics for stretch cases. These are not used as final objective quality scores."""
    mel_hat = torch.as_tensor(mel_hat_cpu, dtype=torch.float32, device=device).unsqueeze(0)
    mel_interp = ctx["mel_interp"].to(device)
    mask = ctx["mask_mel"].to(device)
    mel_hat, mel_interp, mask = trim_to_common_length(mel_hat, mel_interp, mask)
    hf_delta = masked_l1(mel_hat, mel_interp, mask, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).item()
    full_delta = masked_l1(mel_hat, mel_interp, mask, freq_weights=None, mask_dilate=MASK_DILATE).item()
    tdiff_delta = masked_tdiff_l1(mel_hat, mel_interp, mask, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE).item()
    arr = np.asarray(audio_np, dtype=np.float32)
    return dict(
        model=name,
        family=family,
        duration_s=round(len(arr) / SR, 3),
        peak=float(np.max(np.abs(arr))) if arr.size else np.nan,
        rms=float(np.sqrt(np.mean(arr**2))) if arr.size else np.nan,
        delta_from_interp=full_delta,
        hf_delta_from_interp=hf_delta,
        tdiff_delta_from_interp=tdiff_delta,
    )

print("Listening cases:")
for c in EXPANDED_EVAL_CASES:
    print(" ", c["label"])


In [ ]:
# =========================
# Main listening/evaluation loop
# =========================
all_case_rows = []

def _select_specs_by_name(names):
    wanted = set(names)
    return [s for s in resolved_specs if s.get("name") in wanted]

listening_specs = _select_specs_by_name(MAIN_LISTENING_MODEL_NAMES)
print("Listening checkpoint specs:")
display(pd.DataFrame([{
    "name": s["name"],
    "family": s["family"],
    "found": s.get("found", False),
    "ckpt_path": None if s.get("ckpt_path") is None else str(s.get("ckpt_path")),
} for s in listening_specs]))

for case in EXPANDED_EVAL_CASES:
    CLIP_CTX = build_case_context(case)
    K = int(case.get("k", globals().get("K", 1)))
    OFFSET = int(case.get("offset", globals().get("OFFSET", 0)))

    display(Markdown("\n---\n"))
    display(Markdown(f"## {case['label']}"))
    display(Markdown(f"`{CLIP_CTX['path']}`  \n`{CLIP_CTX['display_subtitle']}`"))

    display(Markdown("### References"))
    display(Markdown("**Original waveform**"))
    display(Audio(CLIP_CTX["wav_original"].detach().cpu().numpy(), rate=SR))
    save_audio_if_requested(case["label"], "00_original_waveform", CLIP_CTX["wav_original"].detach().cpu().numpy())

    display(Markdown("**Original mel through BigVGAN**"))
    wav_orig_mel = mel_to_wave_bigvgan(wav_to_bigvgan_mel(CLIP_CTX["wav_original"].unsqueeze(0)))[0]
    display(Audio(wav_orig_mel.detach().cpu().numpy(), rate=SR))
    save_audio_if_requested(case["label"], "01_original_mel_bigvgan", wav_orig_mel.detach().cpu().numpy())

    if case["mode"] == "inpaint":
        display(Markdown("**Linear mel interpolation through BigVGAN**"))
    else:
        display(Markdown(f"**Linear mel stretch baseline through BigVGAN ({CLIP_CTX['speed']:.2f}x)**"))
    wav_interp_mel = mel_to_wave_bigvgan(CLIP_CTX["mel_interp"])[0]
    display(Audio(wav_interp_mel.detach().cpu().numpy(), rate=SR))
    save_audio_if_requested(case["label"], "02_linear_mel_baseline_bigvgan", wav_interp_mel.detach().cpu().numpy())

    rows = []
    if case["mode"] == "inpaint" or CLIP_CTX.get("has_ground_truth", False):
        base_row = exact_summary_row(
            model_name="Linear mel interpolation",
            family="baseline",
            out_kind="BigVGAN",
            mel_hat=CLIP_CTX["mel_interp"],
            mel_real=CLIP_CTX["mel_real"],
            mask_mel=CLIP_CTX["mask_mel"],
            extra={},
        )
        base_row["case"] = case["label"]
        rows.append(base_row)

    display(Markdown("### Model outputs"))
    for spec in listening_specs:
        if not spec.get("found", False):
            display(Markdown(f"**{spec['name']}** - checkpoint not found, skipped."))
            continue
        if spec.get("ckpt_path", None) is None:
            display(Markdown(f"**{spec['name']}** - checkpoint path missing after resolution, skipped."))
            continue

        try:
            display(Markdown(f"#### {spec['name']}"))
            display(Markdown(f"`{spec['ckpt_path']}`"))
            out = run_model(spec)
            display(Audio(out["audio"], rate=SR))
            save_audio_if_requested(case["label"], spec["name"], out["audio"])

            if case["mode"] == "inpaint" or CLIP_CTX.get("has_ground_truth", False):
                row = dict(out["summary"])
                row["case"] = case["label"]
                rows.append(row)
            else:
                row = no_ref_metrics(out["name"], out["family"], out["audio"], out["mel_hat"], CLIP_CTX)
                row["case"] = case["label"]
                rows.append(row)
        except Exception as e:
            print(f"Error while loading/running {spec['name']}: {type(e).__name__}: {e}")

    if rows:
        df = pd.DataFrame(rows)
        all_case_rows.append(df)
        if case["mode"] == "inpaint" or CLIP_CTX.get("has_ground_truth", False):
            display(Markdown("### Controlled inpainting metrics for this listening clip"))
        else:
            display(Markdown("### Stretch diagnostics relative to the linear mel-stretch baseline"))
        display(df)

if all_case_rows:
    expanded_results_df = pd.concat(all_case_rows, ignore_index=True)
    display(Markdown("## Combined listening-case table"))
    display(expanded_results_df)
    csv_path = EVAL_OUT_DIR / "main_listening_results.csv"
    expanded_results_df.to_csv(csv_path, index=False)
    print("Saved combined table to:", csv_path)
else:
    print("No model outputs were collected.")


In [ ]:
# =========================
# Controlled 1x metrics over prepared speech/music test files
# =========================
NUMERIC_OUT_DIR = RUN_DIR / "numeric_metrics"
NUMERIC_OUT_DIR.mkdir(parents=True, exist_ok=True)


def split_paths(split_name: str):
    p = SPLIT_DIR / split_name
    assert p.exists(), f"Missing split file: {p}"
    return [Path(x.strip().strip('"').strip("'")) for x in p.read_text().splitlines() if x.strip()]


def metric_cases_for_split(split_name, source_type, seg_s, max_files):
    paths = split_paths(split_name)
    if max_files is not None:
        paths = paths[: int(max_files)]
    cases = []
    for idx, _ in enumerate(paths):
        cases.append(dict(
            label=f"{source_type.upper()} test idx{idx:04d} 1x inpainting",
            split=split_name,
            index=idx,
            source_type=source_type,
            mode="inpaint",
            k=1,
            offset=0,
            seg_s=float(seg_s),
            start_s=float(METRIC_START_S),
        ))
    return cases


def baseline_metric_row(case):
    row = exact_summary_row(
        model_name="Linear mel interpolation",
        family="baseline",
        out_kind="BigVGAN",
        mel_hat=CLIP_CTX["mel_interp"],
        mel_real=CLIP_CTX["mel_real"],
        mask_mel=CLIP_CTX["mask_mel"],
        extra={},
    )
    row["case"] = case["label"]
    row["split"] = case["split"]
    row["index"] = int(case["index"])
    row["source_type"] = case.get("source_type", "")
    row["start_s"] = float(case.get("start_s", 0.0))
    row["seg_s"] = float(case.get("seg_s", 0.0))
    row["table_group"] = "main"
    return row


def _found_specs_for_names(names):
    wanted = set(names)
    return [s for s in resolved_specs if s.get("found", False) and s.get("name") in wanted]


if RUN_NUMERIC_TEST_EVAL:
    metric_cases = []
    metric_cases += metric_cases_for_split("speech_test.txt", "speech", SPEECH_METRIC_SEG_S, MAX_SPEECH_METRIC_FILES)
    metric_cases += metric_cases_for_split("music_test.txt", "music", MUSIC_METRIC_SEG_S, MAX_MUSIC_METRIC_FILES)
    print("Metric cases:", len(metric_cases))

    main_specs = _found_specs_for_names(MAIN_LISTENING_MODEL_NAMES)
    exploratory_specs = _found_specs_for_names(EXPLORATORY_METRIC_MODEL_NAMES) if RUN_EXPLORATORY_NUMERIC_TEST_EVAL else []
    metric_specs = [(s, "main") for s in main_specs] + [(s, "exploratory") for s in exploratory_specs]
    print("Main metric models:", [s["name"] for s in main_specs])
    print("Exploratory metric models:", [s["name"] for s in exploratory_specs])

    numeric_rows = []
    numeric_errors = []

    for i, case in enumerate(metric_cases, start=1):
        print(f"[{i:04d}/{len(metric_cases):04d}] {case['label']}")
        try:
            CLIP_CTX = build_case_context(case)
            K = int(case.get("k", 1))
            OFFSET = int(case.get("offset", 0))
            numeric_rows.append(baseline_metric_row(case))

            for spec, table_group in metric_specs:
                try:
                    out = run_model(spec)
                    row = dict(out["summary"])
                    row["case"] = case["label"]
                    row["split"] = case["split"]
                    row["index"] = int(case["index"])
                    row["source_type"] = case.get("source_type", "")
                    row["start_s"] = float(case.get("start_s", 0.0))
                    row["seg_s"] = float(case.get("seg_s", 0.0))
                    row["table_group"] = table_group
                    numeric_rows.append(row)
                except Exception as e:
                    numeric_errors.append({
                        **case,
                        "model": spec.get("name", ""),
                        "family": spec.get("family", ""),
                        "table_group": table_group,
                        "error": f"{type(e).__name__}: {e}",
                    })
        except Exception as e:
            numeric_errors.append({**case, "model": "__case_build__", "family": "", "table_group": "", "error": f"{type(e).__name__}: {e}"})

        if (i % SAVE_EVERY_N_CASES) == 0:
            pd.DataFrame(numeric_rows).to_csv(NUMERIC_OUT_DIR / "controlled_1x_metrics_partial.csv", index=False)
            pd.DataFrame(numeric_errors).to_csv(NUMERIC_OUT_DIR / "controlled_1x_errors_partial.csv", index=False)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    numeric_df = pd.DataFrame(numeric_rows)
    numeric_errors_df = pd.DataFrame(numeric_errors)
    numeric_df.to_csv(NUMERIC_OUT_DIR / "controlled_1x_metrics_raw.csv", index=False)
    numeric_errors_df.to_csv(NUMERIC_OUT_DIR / "controlled_1x_errors.csv", index=False)

    metric_cols = [
        "base_recon", "ref_recon", "recon_improvement_pct",
        "base_hf", "ref_hf", "hf_improvement_pct",
        "base_tdiff", "ref_tdiff", "tdiff_improvement_pct",
        "roundtrip_hf", "roundtrip_tdiff",
    ]
    available_metric_cols = [c for c in metric_cols if c in numeric_df.columns]
    grouped = numeric_df.groupby(["table_group", "source_type", "family", "model"], dropna=False)[available_metric_cols].agg(["count", "mean", "std", "median"])
    grouped.to_csv(NUMERIC_OUT_DIR / "controlled_1x_metrics_by_group_source_model.csv")
    display(grouped)

    main_df = numeric_df[numeric_df["table_group"].eq("main")].copy()
    main_grouped = main_df.groupby(["source_type", "family", "model"], dropna=False)[available_metric_cols].agg(["count", "mean", "std", "median"])
    main_grouped.to_csv(NUMERIC_OUT_DIR / "controlled_1x_main_metrics_by_source_model.csv")
    display(Markdown("## Main metric table"))
    display(main_grouped)

    if RUN_EXPLORATORY_NUMERIC_TEST_EVAL:
        exploratory_df = numeric_df[numeric_df["table_group"].eq("exploratory")].copy()
        exploratory_grouped = exploratory_df.groupby(["source_type", "family", "model"], dropna=False)[available_metric_cols].agg(["count", "mean", "std", "median"])
        exploratory_grouped.to_csv(NUMERIC_OUT_DIR / "controlled_1x_exploratory_metrics_by_source_model.csv")
        display(Markdown("## Exploratory metric table"))
        display(exploratory_grouped)

    display(Markdown(f"Saved numeric metrics to `{NUMERIC_OUT_DIR}`"))
    if len(numeric_errors_df):
        display(Markdown("### Numeric evaluation errors"))
        display(numeric_errors_df.head(50))
else:
    print("RUN_NUMERIC_TEST_EVAL=False")


In [ ]:

# =========================
# Plot training logs
# =========================
def load_train_log(spec):
    p = spec.get("log_path", None)
    if p is None:
        return None
    p = Path(p)
    if not p.exists():
        return None
    try:
        df = pd.read_csv(p)
    except Exception as e:
        print(f"Could not read train log for {spec['name']}: {e}")
        return None

    # convert obvious numeric cols
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c], errors="ignore")
        except Exception:
            pass

    if TRAIN_SMOOTH_WINDOW and TRAIN_SMOOTH_WINDOW > 1:
        numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        for c in numeric_cols:
            if c != "step":
                df[c] = df[c].rolling(TRAIN_SMOOTH_WINDOW, min_periods=1).mean()
    return df

if PLOT_TRAINING_LOGS:
    log_frames = {}
    for spec in resolved_specs:
        if not spec["found"]:
            continue
        df = load_train_log(spec)
        if df is not None and len(df) > 0:
            log_frames[spec["name"]] = df

    if len(log_frames) == 0:
        display(Markdown("## Training logs\nNo readable `train_log.csv` files were found for the enabled models."))
    else:
        display(Markdown("## Training-log summaries"))
        log_summary_rows = []
        for spec in resolved_specs:
            if spec["name"] in log_frames:
                df = log_frames[spec["name"]]
                row = dict(
                    model=spec["name"],
                    family=spec["family"],
                    rows=len(df),
                    log_path=str(spec["log_path"]),
                    step_min=df["step"].min() if "step" in df.columns else None,
                    step_max=df["step"].max() if "step" in df.columns else None,
                )
                log_summary_rows.append(row)
        display(pd.DataFrame(log_summary_rows))

        metrics = []
        for m in TRAIN_METRICS_TO_PLOT:
            if any(m in df.columns for df in log_frames.values()):
                metrics.append(m)

        if len(metrics) == 0:
            display(Markdown("No requested training metrics were found in the available logs."))
        else:
            ncols = TRAIN_PLOT_NCOLS
            nrows = int(np.ceil(len(metrics) / ncols))
            fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 3.6 * nrows), squeeze=False)
            axes = axes.ravel()

            for ax, metric in zip(axes, metrics):
                plotted = False
                for spec in resolved_specs:
                    name = spec["name"]
                    if name not in log_frames:
                        continue
                    df = log_frames[name]
                    if metric not in df.columns:
                        continue
                    x = df["step"] if "step" in df.columns else np.arange(len(df))
                    y = df[metric]
                    if not pd.api.types.is_numeric_dtype(y):
                        continue
                    ax.plot(x, y, label=name, linewidth=1.8)
                    plotted = True
                ax.set_title(metric)
                ax.set_xlabel("step")
                ax.grid(alpha=0.25)
                if plotted:
                    ax.legend(fontsize=8)
                else:
                    ax.set_visible(False)

            for ax in axes[len(metrics):]:
                ax.set_visible(False)

            plt.tight_layout()
            plt.show()
